# MagCarto: Earth's Magnetosphere Visualization

## Interactive Notebook for Magnetosphere Boundary Modeling

This notebook runs all MagCarto scripts with interactive date/time selection.
Visualizes:
- **Magnetopause** (Shue 1998 empirical model)
- **Bow Shock** (Mach number dependent)
- **Magnetic field lines** (T96+IGRF tracing)
- **Wind spacecraft trajectory** (NASA SSCWeb)

---

In [1]:
# Setup and imports
import sys
import os
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt
import warnings
import time
import pickle
from pathlib import Path
warnings.filterwarnings('ignore')

# Add scripts folder to path
sys.path.insert(0, os.path.join(os.getcwd(), 'scripts'))

# SpacePy and geophysics libraries
import spacepy.time as spt
import spacepy.omni as om
import spacepy.toolbox as tb
import spacepy.coordinates as spc
from geopack import geopack as gp
from sscws.sscws import SscWs

# Widgets for interactivity
from ipywidgets import widgets, interactive, Output, VBox, HBox, Label
from IPython.display import display, clear_output

print("✓ All imports successful!")
print(f"Working directory: {os.getcwd()}")

Load IGRF coefficients ...
✓ All imports successful!
Working directory: C:\Users\mmartinovic\OneDrive - University of Arizona\00_Python_Scripts\QTN\Wind_Mass_Filtering\bow_shock\MagCarto


In [2]:
# Constants and helper functions
RE_KM = 6371.2  # Earth radius in km
os.makedirs('figs', exist_ok=True)

def unix_seconds(t_utc: dt.datetime) -> float:
    """Convert datetime to Unix seconds."""
    if t_utc.tzinfo is None:
        t_utc = t_utc.replace(tzinfo=dt.timezone.utc)
    epoch = dt.datetime(1970, 1, 1, tzinfo=dt.timezone.utc)
    return (t_utc - epoch).total_seconds()

def _parse_times(time_field):
    """Accept list/array of datetime objects, ISO strings, or epoch seconds."""
    if isinstance(time_field, (dt.datetime, str, int, float)):
        time_list = [time_field]
    else:
        time_list = list(time_field)

    out = []
    for t in time_list:
        if isinstance(t, dt.datetime):
            if t.tzinfo is None:
                t = t.replace(tzinfo=dt.timezone.utc)
            out.append(t)
        elif isinstance(t, str):
            # Clean up: remove Z since isoformat() adds timezone info
            s = str(t).strip()
            if s.endswith('Z'):
                s = s[:-1]
            try:
                parsed = dt.datetime.fromisoformat(s)
                if parsed.tzinfo is None:
                    parsed = parsed.replace(tzinfo=dt.timezone.utc)
                out.append(parsed)
            except ValueError as e:
                raise ValueError(f"Could not parse time string: {t}") from e
        elif isinstance(t, (int, float)):
            out.append(dt.datetime.fromtimestamp(t, tz=dt.timezone.utc))
        else:
            raise TypeError(f"Unrecognized time element type: {type(t)} value={t}")
    return out

print("✓ Helper functions loaded")

✓ Helper functions loaded


In [3]:
# Load OMNI2 cache and data retrieval functions

def load_omni2_cache():
    """Load OMNI2 data from local cache."""
    cache_file = Path("D:\\Data\\OMNI\\omni2_cache.pkl")
    if cache_file.exists():
        try:
            with open(cache_file, 'rb') as f:
                omni_cache = pickle.load(f)
            print(f"✓ Loaded OMNI2 cache from {cache_file}")
            return omni_cache
        except Exception as e:
            print(f"⚠️  Failed to load cache: {e}")
            return None
    else:
        print(f"⚠️  Cache not found at {cache_file}")
        return None

# Load cache once at startup
OMNI2_CACHE = load_omni2_cache()

def get_omni2_params(t_iso: str):
    """Fetch OMNI2 solar wind parameters for a given time.
    First tries local cache, then falls back to remote fetch.
    """
    global OMNI2_CACHE

    # Try cache first
    if OMNI2_CACHE:
        try:
            t_req = dt.datetime.fromisoformat(t_iso.replace("Z", "+00:00"))
            cache_times = [dt.datetime.fromisoformat(str(t).replace("Z", "+00:00"))
                          if isinstance(t, str) else t
                          for t in OMNI2_CACHE.get("Epoch", [])]
            if cache_times:
                idx = int(np.argmin([abs((ct - t_req).total_seconds()) for ct in cache_times]))
                Pdyn = float(OMNI2_CACHE["Flow_pressure"][idx])
                Dst  = float(OMNI2_CACHE["Dst_index"][idx])
                By   = float(OMNI2_CACHE["By_GSM"][idx])
                Bz   = float(OMNI2_CACHE["Bz_GSM"][idx])
                Ma   = float(OMNI2_CACHE["Alfven_mach_number"][idx])
                print(f"✓ Using cached OMNI2 data for {t_iso}")
                return Pdyn, Dst, By, Bz, Ma
        except Exception as e:
            print(f"⚠️  Cache lookup failed: {e}")

    # Fallback to remote fetch
    print("Fetching OMNI2 from remote server...")
    ticks = spt.Ticktock([t_iso], "ISO")
    try:
        omni = om.get_omni(ticks, dbase="OMNI2hourly")
    except Exception:
        print("Updating OMNI2 database...")
        tb.update(omni2=True)
        omni = om.get_omni(ticks, dbase="OMNI2hourly")

    Pdyn = float(omni["Flow_pressure"][0])
    Dst  = float(omni["Dst_index"][0])
    By   = float(omni["By_GSM"][0])
    Bz   = float(omni["Bz_GSM"][0])
    Ma   = float(omni["Alfven_mach_number"][0])
    return Pdyn, Dst, By, Bz, Ma

def t96_parmod(Pdyn, Dst, By, Bz):
    """Create T96 model parameter array."""
    par = np.zeros(10, dtype=float)
    par[0] = Pdyn
    par[1] = Dst
    par[2] = By
    par[3] = Bz
    return par

def _extract_xyz_any(sat_data):
    """Extract X, Y, Z coordinates from SSCWeb response."""
    coords = sat_data.get("Coordinates")
    if isinstance(coords, dict):
        entries = [coords]
    else:
        entries = list(coords)
    for c in entries:
        if isinstance(c, dict) and all(k in c for k in ("X", "Y", "Z")):
            x = np.asarray(c["X"], float)
            y = np.asarray(c["Y"], float)
            z = np.asarray(c["Z"], float)
            units = c.get("Units") or c.get("units")
            cs = c.get("CoordinateSystem")
            coord_sys = cs.value if hasattr(cs, "value") else (str(cs) if cs is not None else None)
            return x, y, z, units, coord_sys
    raise KeyError(f"Could not find X/Y/Z in Coordinates")

def get_wind_gsm_from_ssc(start_iso, stop_iso, observatory_id="wind"):
    """Fetch Wind spacecraft ephemeris from SSCWeb and convert to GSM coordinates."""
    print(f"Fetching {observatory_id.upper()} ephemeris from SSCWeb...")
    ssc = SscWs()
    result = ssc.get_locations([observatory_id], [start_iso, stop_iso])
    if "Data" not in result or not result.get("Data"):
        msg = "\n".join([f"{k}: {result.get(k)}" for k in ["HttpStatus", "ErrorMessage", "ErrorDescription"]])
        raise RuntimeError(f"SSCWeb returned no Data.\n{msg}")
    sat_data = result["Data"][0]
    times = _parse_times(sat_data["Time"])
    x, y, z, units, coord_sys = _extract_xyz_any(sat_data)
    if units is None:
        med = np.nanmedian(np.abs(x))
        units = "km" if med > 1000 else "Re"
    if units.lower() in ["km", "kilometer", "kilometers"]:
        x = x / RE_KM
        y = y / RE_KM
        z = z / RE_KM
    xyz_gse = np.vstack([x, y, z]).T
    tt = spt.Ticktock(times, "UTC")
    c_gse = spc.Coords(xyz_gse, "GSE", "car", ticks=tt)
    c_gsm = c_gse.convert("GSM", "car")
    xyz_gsm = np.asarray(c_gsm.data)
    print(f"✓ Retrieved {len(times)} {observatory_id.upper()} positions")
    return times, xyz_gsm

print("✓ Data retrieval functions loaded")

⚠️  Cache not found at D:\Data\OMNI\omni2_cache.pkl
✓ Data retrieval functions loaded


In [4]:
# Magnetosphere boundary models

def shue98_magnetopause_rtheta(Pdyn, Bz, ntheta=361):
    """Shue et al. (1998) empirical magnetopause model."""
    if not np.isfinite(Pdyn) or Pdyn <= 0:
        raise ValueError(f"Nonpositive/invalid Pdyn={Pdyn}")
    if not np.isfinite(Bz):
        raise ValueError(f"Invalid Bz={Bz}")
    r0 = (10.22 + 1.29 * np.tanh(0.184 * (Bz + 8.14))) * (Pdyn ** (-1.0 / 6.6))
    alpha = (0.58 - 0.007 * Bz) * (1.0 + 0.024 * np.log(Pdyn))
    eps = 1e-6
    theta = np.linspace(0, np.pi - eps, ntheta)
    r = r0 * (2.0 / (1.0 + np.cos(theta))) ** alpha
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    return x, y, r0, alpha

def bowshock_rtheta_from_Ma(r0_mp, Ma, ntheta=361, e=0.9, gamma=5/3, delta_min=1.0, delta_max=10.0):
    """Conic bow shock model dependent on Alfvén Mach number."""
    if (not np.isfinite(Ma)) or (Ma <= 1.0):
        delta = 2.5
    else:
        M2 = Ma * Ma
        frac = ((gamma - 1.0) * M2 + 2.0) / ((gamma + 1.0) * (M2 - 1.0))
        delta = float(np.clip(r0_mp * frac, delta_min, delta_max))
    Rbs0 = r0_mp + delta
    eps = 1e-6
    theta = np.linspace(0.0, np.pi - eps, ntheta)
    r = Rbs0 * (1.0 + e) / (1.0 + e * np.cos(theta))
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    return x, y, Rbs0, delta

print("✓ Magnetopause & bow shock models loaded")

✓ Magnetopause & bow shock models loaded


In [5]:
# Field line tracing

def trace_field_line(seed_xyz_gsm, parmod, ut_seconds, rlim=60.0, r0=1.0):
    """Trace magnetic field lines in both directions using geopack (T96+IGRF)."""
    ps = gp.recalc(ut_seconds)
    x0, y0, z0 = seed_xyz_gsm
    xf1, yf1, zf1, xx1, yy1, zz1 = gp.trace(x0, y0, z0, -1, rlim, r0, parmod, "t96", "igrf")
    xf2, yf2, zf2, xx2, yy2, zz2 = gp.trace(x0, y0, z0,  1, rlim, r0, parmod, "t96", "igrf")
    X = np.concatenate([np.asarray(xx1)[::-1], np.asarray(xx2)[1:]])
    Y = np.concatenate([np.asarray(yy1)[::-1], np.asarray(yy2)[1:]])
    Z = np.concatenate([np.asarray(zz1)[::-1], np.asarray(zz2)[1:]])
    return X, Y, Z, ps

print("✓ Field line tracing function loaded")

✓ Field line tracing function loaded


## Advanced: Self-Consistent Magnetopause (T96+IGRF Last-Closed)

This section computes the magnetopause using field line tracing instead of empirical models. It's slower but more physically accurate.

In [6]:
def _trace_end_radius(x0, y0, z0, direction, rlim, r0, parmod):
    """Trace field line and return radius of final point."""
    xf, yf, zf, xx, yy, zz = gp.trace(x0, y0, z0, direction, rlim, r0, parmod, "t96", "igrf")
    if not np.isfinite(xf) or not np.isfinite(yf) or not np.isfinite(zf):
        return np.inf
    return float(np.sqrt(xf*xf + yf*yf + zf*zf))

def is_closed_fieldline(x0, y0, z0, ut_seconds, parmod, rlim=60.0, r0=1.0, rtol=0.05):
    """Test if a field line is closed (both ends hit Earth) or open."""
    gp.recalc(ut_seconds)
    r_end_1 = _trace_end_radius(x0, y0, z0, +1, rlim, r0, parmod)
    r_end_2 = _trace_end_radius(x0, y0, z0, -1, rlim, r0, parmod)
    hits_earth_1 = (r_end_1 <= (r0 + rtol))
    hits_earth_2 = (r_end_2 <= (r0 + rtol))
    return bool(hits_earth_1 and hits_earth_2)

def find_boundary_on_ray(theta, plane, ut_seconds, parmod,
                         r_min=1.2, r_max=60.0, dr=0.5,
                         rlim=60.0, r0=1.0, rtol=0.05,
                         bisect_tol=0.05, max_bisect=25,
                         last_r_guess=None):
    """Find magnetopause boundary radius on a ray via bisection."""
    ct, st = np.cos(theta), np.sin(theta)
    if plane == "xy":
        def point(r):
            return (r*ct, r*st, 0.0)
    elif plane == "xz":
        def point(r):
            return (r*ct, 0.0, r*st)
    else:
        raise ValueError("plane must be 'xy' or 'xz'")
    
    if last_r_guess is not None and np.isfinite(last_r_guess):
        r_start = max(r_min, last_r_guess - 2.0)
    else:
        r_start = r_min
    
    r = r_start
    x, y, z = point(r)
    closed = is_closed_fieldline(x, y, z, ut_seconds, parmod, rlim=rlim, r0=r0, rtol=rtol)
    
    if not closed:
        for _ in range(10):
            r_try = max(r_min, r - dr)
            if r_try == r:
                break
            r = r_try
            x, y, z = point(r)
            closed = is_closed_fieldline(x, y, z, ut_seconds, parmod, rlim=rlim, r0=r0, rtol=rtol)
            if closed:
                break
    
    if not closed:
        return np.nan, False
    
    r_lo = r
    r_hi = None
    r = r_lo
    while r < r_max:
        r_next = r + dr
        x, y, z = point(r_next)
        closed_next = is_closed_fieldline(x, y, z, ut_seconds, parmod, rlim=rlim, r0=r0, rtol=rtol)
        if not closed_next:
            r_hi = r_next
            break
        r = r_next
        r_lo = r
    
    if r_hi is None:
        return np.nan, False
    
    a, b = r_lo, r_hi
    for _ in range(max_bisect):
        m = 0.5*(a + b)
        x, y, z = point(m)
        closed_m = is_closed_fieldline(x, y, z, ut_seconds, parmod, rlim=rlim, r0=r0, rtol=rtol)
        if closed_m:
            a = m
        else:
            b = m
        if (b - a) <= bisect_tol:
            break
    
    r_boundary = 0.5*(a + b)
    return float(r_boundary), True

print("✓ Advanced T96 magnetopause functions loaded")

✓ Advanced T96 magnetopause functions loaded


In [7]:
from ipywidgets import widgets

print("Setting up advanced T96 visualization...")

run_advanced_button = widgets.Button(
    description='Generate T96 Self-Consistent Plot',
    button_style='success',
    tooltip='Click to generate advanced T96+IGRF magnetopause visualization (slower)',
    icon='star'
)

advanced_output_box = widgets.Output()

def on_run_advanced_clicked(b):
    with advanced_output_box:
        clear_output(wait=True)
        try:
            # Parse selected datetime
            year, month, day = date_picker.value.year, date_picker.value.month, date_picker.value.day
            hour, minute = hour_slider.value, minute_slider.value
            
            t_iso = f"{year:04d}-{month:02d}-{day:02d}T{hour:02d}:{minute:02d}:00"
            t0 = dt.datetime(year, month, day, hour, minute, 0, tzinfo=dt.timezone.utc)
            ut = unix_seconds(t0)
            
            print(f"📅 Running T96 self-consistent visualization for {t_iso}Z")
            print("="*60)
            print("⚠️  Note: This may take 2-5 minutes due to field line tracing...")
            print()
            
            # Fetch OMNI2 parameters
            print("1️⃣  Fetching solar wind parameters (OMNI2)...")
            Pdyn, Dst, By, Bz, Ma = get_omni2_params(t_iso)
            parmod = t96_parmod(Pdyn, Dst, By, Bz)
            print(f"   ✓ Pdyn={Pdyn:.2f} nPa, Bz={Bz:.2f} nT, Ma={Ma:.2f}")
            
            # Compute empirical boundaries
            print("\n2️⃣  Computing empirical models for comparison...")
            xmp, ymp, r0mp, alphamp = shue98_magnetopause_rtheta(Pdyn, Bz, ntheta=361)
            xbs, ybs, rbs0, delta_bs = bowshock_rtheta_from_Ma(r0mp, Ma, ntheta=361)
            
            # Compute T96 magnetopause (this takes time)
            print("\n3️⃣  Computing T96+IGRF magnetopause via last-closed field lines...")
            print("   Computing XY plane (equatorial)...")
            thetas_xy = np.linspace(0.0, np.pi, 91)
            x_list_xy, y_list_xy = [], []
            r_prev = None
            
            for i, th in enumerate(thetas_xy):
                if (i+1) % 15 == 0:
                    print(f"   ... {i+1}/{len(thetas_xy)} angles...")
                r_b, ok = find_boundary_on_ray(
                    th, "xy", ut, parmod,
                    r_min=1.2, r_max=60.0, dr=0.6,
                    bisect_tol=0.05,
                    last_r_guess=r_prev
                )
                if ok and np.isfinite(r_b):
                    ct, st = np.cos(th), np.sin(th)
                    x_list_xy.append(r_b*ct)
                    y_list_xy.append(r_b*st)
                    r_prev = r_b
            
            xmp_xy, ymp_xy = np.array(x_list_xy), np.array(y_list_xy)
            
            print("   Computing XZ plane (meridian)...")
            thetas_xz = np.linspace(0.0, np.pi, 91)
            x_list_xz, z_list_xz = [], []
            r_prev = None
            
            for i, th in enumerate(thetas_xz):
                if (i+1) % 15 == 0:
                    print(f"   ... {i+1}/{len(thetas_xz)} angles...")
                r_b, ok = find_boundary_on_ray(
                    th, "xz", ut, parmod,
                    r_min=1.2, r_max=60.0, dr=0.6,
                    bisect_tol=0.05,
                    last_r_guess=r_prev
                )
                if ok and np.isfinite(r_b):
                    ct, st = np.cos(th), np.sin(th)
                    x_list_xz.append(r_b*ct)
                    z_list_xz.append(r_b*st)
                    r_prev = r_b
            
            xmp_xz, zmp_xz = np.array(x_list_xz), np.array(z_list_xz)
            print("   ✓ T96 magnetopause computed")
            
            # Fetch Wind
            print("\n4️⃣  Fetching Wind spacecraft ephemeris...")
            start = (t0 - dt.timedelta(hours=6)).isoformat()
            stop = (t0 + dt.timedelta(hours=6)).isoformat()
            times_w, xyz_w = get_wind_gsm_from_ssc(start, stop, observatory_id="wind")
            i0 = int(np.argmin([abs((ti - t0).total_seconds()) for ti in times_w]))
            
            # Create plots
            print("\n5️⃣  Generating comparison plots...")
            fig, axs = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
            ax_xy, ax_xz = axs
            
            # XY plane
            ax_xy.plot(xmp_xy, ymp_xy, "-.", lw=2.5, color='#32a852', label="MP (T96+IGRF, last-closed)")
            ax_xy.plot(xmp_xy, -ymp_xy, "-.", lw=2.5, color='#32a852')
            ax_xy.plot(xmp, ymp, "-.", lw=1.5, color='#a0a0a0', label="Shue 1998 (empirical)", alpha=0.6)
            ax_xy.plot(xmp, -ymp, "-.", lw=1.5, color='#a0a0a0', alpha=0.6)
            ax_xy.plot(xbs, ybs, "--", lw=2, color='#a0a0a0', label="Bow shock", alpha=0.6)
            ax_xy.plot(xbs, -ybs, "--", lw=2, color='#a0a0a0', alpha=0.6)
            ax_xy.plot(xyz_w[:, 0], xyz_w[:, 1], lw=2, color='#c01d14', label="Wind")
            ax_xy.plot(xyz_w[i0, 0], xyz_w[i0, 1], "o", ms=8, color='#fc0394')
            ax_xy.plot(0, 0, "o", ms=6, color='black')
            ax_xy.set_aspect("equal", adjustable="box")
            ax_xy.set_xlabel(r"$X_{GSM} [R_E]$", fontsize=11)
            ax_xy.set_ylabel(r"$Y_{GSM} [R_E]$", fontsize=11)
            ax_xy.set_title("Equatorial Plane (XY)", fontsize=12, fontweight='bold')
            ax_xy.grid(True, alpha=0.3)
            ax_xy.set_xlim(-35, 35)
            ax_xy.set_ylim(-35, 35)
            ax_xy.legend(loc="upper left", frameon=False)
            
            # XZ plane
            ax_xz.plot(xmp_xz, zmp_xz, "-.", lw=2.5, color='#32a852', label="MP (T96+IGRF)")
            ax_xz.plot(xmp_xz, -zmp_xz, "-.", lw=2.5, color='#32a852')
            ax_xz.plot(xmp, ymp, "-.", lw=1.5, color='#a0a0a0', alpha=0.6)
            ax_xz.plot(xmp, -ymp, "-.", lw=1.5, color='#a0a0a0', alpha=0.6)
            ax_xz.plot(xbs, ybs, "--", lw=2, color='#a0a0a0', alpha=0.6)
            ax_xz.plot(xbs, -ybs, "--", lw=2, color='#a0a0a0', alpha=0.6)
            ax_xz.plot(xyz_w[:, 0], xyz_w[:, 2], lw=2, color='#c01d14', label="Wind")
            ax_xz.plot(xyz_w[i0, 0], xyz_w[i0, 2], "o", ms=8, color='#fc0394')
            th = np.linspace(0, 2*np.pi, 361)
            ax_xz.plot(np.cos(th), np.sin(th), lw=2, color='#8B4513')
            ax_xz.set_aspect("equal", adjustable="box")
            ax_xz.set_xlabel(r"$X_{GSM} [R_E]$", fontsize=11)
            ax_xz.set_ylabel(r"$Z_{GSM} [R_E]$", fontsize=11)
            ax_xz.set_title("Meridian Plane (XZ)", fontsize=12, fontweight='bold')
            ax_xz.grid(True, alpha=0.3)
            ax_xz.set_xlim(-35, 35)
            ax_xz.set_ylim(-35, 35)
            ax_xz.legend(loc="upper left", frameon=False)
            
            # Title
            fig.suptitle(
                f"{t_iso}Z | OMNI2: Pdyn={Pdyn:.2f} nPa, Dst={Dst:.0f} nT, By={By:.2f} nT, Bz={Bz:.2f} nT, M_A={Ma:.2f}\n"
                f"Green: T96+IGRF Self-Consistent | Gray: Empirical Models",
                fontsize=10,
                fontweight='bold'
            )
            
            # Save
            outpng = f"figs/T96_Magnetosphere_wind_{year:04d}-{month:02d}-{day:02d}T{hour:02d}{minute:02d}Z.png"
            fig.savefig(outpng, dpi=150, bbox_inches='tight')
            print(f"   ✓ Saved to {outpng}")
            
            plt.show()
            
            print("\n" + "="*60)
            print("✅ T96 visualization complete!")
            
        except Exception as e:
            print(f"❌ Error: {type(e).__name__}: {e}")
            import traceback
            traceback.print_exc()

run_advanced_button.on_click(on_run_advanced_clicked)

# Display button and output (NO VBox)
display(widgets.Label("🚀 Advanced: T96+IGRF Self-Consistent Magnetopause (slow)"))
display(run_advanced_button)
display(advanced_output_box)

print("✓ Advanced T96 magnetopause functions loaded")

Setting up advanced T96 visualization...


Label(value='🚀 Advanced: T96+IGRF Self-Consistent Magnetopause (slow)')

Button(button_style='success', description='Generate T96 Self-Consistent Plot', icon='star', style=ButtonStyle…

Output()

✓ Advanced T96 magnetopause functions loaded


In [8]:
"""
NOTEBOOK CELL: Interactive 3D Pass + Direction Magnetosphere Viewer (Plotly)

PURPOSE:
Select a bow shock crossing pass and direction, view interactive 3D magnetosphere
with all crossings for that configuration, including time windows and field lines.

FEATURES:
- Pass ID dropdown (69 passes available)
- Direction dropdown (Inbound or Outbound)
- Real-time OMNI2 solar wind parameter fetching with 7-attempt retry
- Shue 1998 magnetopause + bow shock model
- T96+IGRF magnetic field line tracing
- Wind spacecraft trajectory with crossing markers
- Broad and zoom time windows highlighted on trajectory
- Full 3D interactive visualization using Plotly:
  * Click & drag to rotate
  * Scroll to zoom
  * Right-click & drag to pan
  * Hover over crossings for timestamp tooltips

REQUIREMENTS:
- plotly package installed
- bs_crossings_loader module in same directory
- All magnetosphere functions (get_omni2_params, t96_parmod, etc.) pre-loaded

INTEGRATION:
Copy this entire cell into MagCarto_Master.ipynb after all helper function definitions
and after bs_crossings_loader import.
"""

print("Loading bow shock crossings database...\n")

from bs_crossings_loader import load_crossings
from ipywidgets import widgets, Output
from IPython.display import display, clear_output
import datetime as dt
import plotly.graph_objects as go
import numpy as np
import time

# Load database
db = load_crossings()
print(f"✓ Loaded {len(db)} crossings in {len(db.get_passes())} passes\n")

# Get list of passes
all_passes = db.get_passes()

# ============================================================================
# Create widgets
# ============================================================================

pass_dropdown = widgets.Dropdown(
    options=[(f"Pass {p}", p) for p in all_passes],
    value=all_passes[0],
    description='Pass ID:',
    style={'description_width': '100px'}
)

direction_dropdown = widgets.Dropdown(
    options=[('Inbound (entering bow shock)', 'in'),
             ('Outbound (leaving bow shock)', 'out')],
    value='in',
    description='Direction:',
    style={'description_width': '100px'}
)

run_button = widgets.Button(
    description='Generate Interactive 3D Visualization',
    button_style='info',
    icon='cube',
    tooltip='Generate full 3D magnetosphere for selected pass & direction'
)

info_output = Output()
viz_output = Output()

# ============================================================================
# Event handlers
# ============================================================================

def on_selection_changed(change):
    """Show crossing info when pass or direction is selected."""
    pass_id = pass_dropdown.value
    direction = direction_dropdown.value

    all_crossings = db.get_crossings_for_pass(pass_id)
    crossings = [c for c in all_crossings if c.crossing_direction == direction]

    with info_output:
        clear_output(wait=True)
        direction_name = "Inbound" if direction == 'in' else "Outbound"
        print(f"📍 Pass {pass_id} - {direction_name} Crossings: {len(crossings)}")

        if len(crossings) == 0:
            print(f"\n⚠️  No {direction_name.lower()} crossings in this pass!")
            return

        # Show broad/zoom windows from first crossing that has them
        print()
        for c in crossings:
            if c.has_broad_window():
                print(f"     Broad window:  {c.broad_start.strftime('%Y-%m-%d %H:%M:%S')} to {c.broad_end.strftime('%Y-%m-%d %H:%M:%S')}")
                break

        for c in crossings:
            if c.has_zoom_window():
                print(f"     Zoom window:   {c.zoom_start.strftime('%Y-%m-%d %H:%M:%S')} to {c.zoom_end.strftime('%Y-%m-%d %H:%M:%S')}")
                break

        if crossings:
            print(f"     Time range: {min(c.time for c in crossings).strftime('%Y-%m-%d %H:%M:%S')} to "
                  f"{max(c.time for c in crossings).strftime('%Y-%m-%d %H:%M:%S')}")

        print("-" * 80)
        for i, c in enumerate(crossings, 1):
            print(f"  {i}. Crossing: {c.time.strftime('%Y-%m-%d %H:%M:%S')}")

def on_run_clicked(b):
    """Generate interactive 3D magnetosphere visualization for selected pass & direction."""
    with viz_output:
        clear_output(wait=True)

        try:
            # Get selected pass and direction
            pass_id = pass_dropdown.value
            direction = direction_dropdown.value
            direction_name = "Inbound" if direction == 'in' else "Outbound"

            all_crossings = db.get_crossings_for_pass(pass_id)
            crossings = [c for c in all_crossings if c.crossing_direction == direction]

            if not crossings:
                print(f"❌ No {direction_name.lower()} crossings in Pass {pass_id}!")
                return

            # Use FIRST crossing with a zoom window
            t0 = None
            for c in crossings:
                if c.has_zoom_window():
                    t0 = c.zoom_start + (c.zoom_end - c.zoom_start) / 2
                    print(f"Using time from first {direction_name.lower()} crossing with zoom window")
                    break

            if t0 is None:
                t0 = crossings[0].time
                print(f"Using time from first {direction_name.lower()} crossing")

            year = t0.year
            month = t0.month
            day = t0.day
            hour = t0.hour
            minute = t0.minute
            t_iso = f"{year:04d}-{month:02d}-{day:02d}T{hour:02d}:{minute:02d}:00"

            print(f"📅 Running interactive 3D magnetosphere visualization for Pass {pass_id} ({direction_name})")
            print(f"   Reference time: {t_iso}Z")
            print(f"   Showing {len(crossings)} {direction_name.lower()} crossing(s)")
            print("="*80)

            # Fetch OMNI2 with progressive time window expansion
            print("\n1️⃣  Fetching solar wind parameters (OMNI2)...")

            time_windows = [
                (0, "exact timestamp"),
                (10, "±10 minutes"),
                (30, "±30 minutes"),
                (60, "±1 hour"),
                (120, "±2 hours"),
                (240, "±4 hours"),
                (480, "±8 hours"),
            ]

            Pdyn, Dst, By, Bz, Ma = None, None, None, None, None
            successful_window = None
            last_error = None

            for window_minutes, window_desc in time_windows:
                try:
                    print(f"   Trying {window_desc}...", end=" ")

                    if window_minutes == 0:
                        # Try exact timestamp
                        Pdyn, Dst, By, Bz, Ma = get_omni2_params(t_iso)
                    else:
                        # Try time window - average multiple points
                        t_start = t0 - dt.timedelta(minutes=window_minutes)
                        t_end = t0 + dt.timedelta(minutes=window_minutes)
                        t_start_iso = t_start.isoformat()
                        t_end_iso = t_end.isoformat()

                        # Try fetching from start, middle, and end of window
                        values = []
                        for t_fetch in [t_start, t0, t_end]:
                            try:
                                vals = get_omni2_params(t_fetch.isoformat())
                                if all(np.isfinite(v) for v in vals):
                                    values.append(vals)
                            except:
                                pass

                        if not values:
                            raise ValueError(f"No valid data in {window_desc}")

                        # Average the valid values
                        Pdyn = np.mean([v[0] for v in values])
                        Dst = np.mean([v[1] for v in values])
                        By = np.mean([v[2] for v in values])
                        Bz = np.mean([v[3] for v in values])
                        Ma = np.mean([v[4] for v in values])

                    # Validate data
                    if not np.isfinite(Pdyn) or not np.isfinite(Bz):
                        raise ValueError(f"OMNI2 returned NaN values")

                    print(f"✓ Success!")
                    successful_window = window_desc
                    break

                except Exception as e:
                    last_error = e
                    print(f"✗ Failed: {type(e).__name__}")
                    time.sleep(0.5)

            if Pdyn is None or not np.isfinite(Pdyn):
                raise Exception(f"Failed to fetch valid OMNI2 data: {last_error}")

            print(f"   ✓ Using data from {successful_window}")

            parmod = t96_parmod(Pdyn, Dst, By, Bz)
            print(f"   ✓ Pdyn={Pdyn:.2f} nPa, Bz={Bz:.2f} nT, Ma={Ma:.2f}")

            # Compute boundaries
            print("\n2️⃣  Computing magnetopause (Shue 1998) and bow shock...")
            xmp, ymp, r0mp, alphamp = shue98_magnetopause_rtheta(Pdyn, Bz, ntheta=361)
            xbs, ybs, rbs0, delta_bs = bowshock_rtheta_from_Ma(r0mp, Ma, ntheta=361)
            print(f"   ✓ MP radius: {r0mp:.2f} Re, BS distance: {delta_bs:.2f} Re")

            # Trace field lines - distributed in 3D space
            print("\n3️⃣  Tracing magnetic field lines (T96+IGRF)...")
            seeds = []

            # Seeds distributed around a sphere at different latitudes
            r_seed = 2.5
            n_lon = 12  # Longitude (phi) points
            n_lat = 6   # Latitude (theta) points

            for lat_idx, lat in enumerate(np.linspace(-np.pi/2, np.pi/2, n_lat)):
                for lon_idx, lon in enumerate(np.linspace(0, 2*np.pi, n_lon, endpoint=False)):
                    x = r_seed * np.cos(lat) * np.cos(lon)
                    y = r_seed * np.cos(lat) * np.sin(lon)
                    z = r_seed * np.sin(lat)
                    seeds.append((x, y, z))

            # Add tail seeds
            for r in [3.5, 5.0]:
                for angle in np.linspace(0, np.pi/2, 6):
                    for sign in [1, -1]:
                        x = -r * np.cos(angle)
                        y = 0
                        z = sign * r * np.sin(angle)
                        seeds.append((x, y, z))

            field_lines = []
            ps_last = None
            ut = unix_seconds(t0)

            for i, s in enumerate(seeds):
                X, Y, Z, ps = trace_field_line(s, parmod, ut, rlim=60.0, r0=1.0)
                field_lines.append((X, Y, Z))
                ps_last = ps
                if (i+1) % 10 == 0:
                    print(f"   ✓ Traced {i+1}/{len(seeds)} field lines...")

            # Fetch Wind
            print("\n4️⃣  Fetching Wind spacecraft ephemeris (24h window)...")
            start = (t0 - dt.timedelta(hours=12)).isoformat()
            stop = (t0 + dt.timedelta(hours=12)).isoformat()
            times_w, xyz_w = get_wind_gsm_from_ssc(start, stop, observatory_id="wind")
            i0 = int(np.argmin([abs((ti - t0).total_seconds()) for ti in times_w]))

            # Create 3D Plotly figure
            print("\n5️⃣  Generating interactive 3D plot...")
            fig = go.Figure()

            # Create magnetopause surface as mesh without visible gridlines
            angles_mp = np.linspace(0, 2*np.pi, 100)
            xmp_flat = []
            ymp_flat = []
            zmp_flat = []

            for angle in angles_mp:
                xmp_flat.extend(xmp)
                ymp_flat.extend(ymp * np.cos(angle))
                zmp_flat.extend(ymp * np.sin(angle))

            # Create face indices for triangular mesh
            n_angles = len(angles_mp)
            n_radii = len(xmp)
            faces_i = []
            faces_j = []
            faces_k = []

            for i in range(n_angles - 1):
                for j in range(n_radii - 1):
                    # First triangle
                    faces_i.append(i * n_radii + j)
                    faces_j.append((i + 1) * n_radii + j)
                    faces_k.append(i * n_radii + (j + 1))
                    # Second triangle
                    faces_i.append((i + 1) * n_radii + j)
                    faces_j.append((i + 1) * n_radii + (j + 1))
                    faces_k.append(i * n_radii + (j + 1))

            fig.add_trace(go.Mesh3d(
                x=xmp_flat, y=ymp_flat, z=zmp_flat,
                i=faces_i, j=faces_j, k=faces_k,
                color='#326ba8',
                name='Magnetopause',
                opacity=0.35,
                showlegend=True,
                hoverinfo='skip'
            ))

            # Create bow shock surface by rotating 2D profile
            angles_bs = np.linspace(0, 2*np.pi, 80)
            xbs_surf = np.tile(xbs, (len(angles_bs), 1))
            ybs_surf = np.zeros((len(angles_bs), len(xbs)))
            zbs_surf = np.zeros((len(angles_bs), len(xbs)))

            for i, angle in enumerate(angles_bs):
                ybs_surf[i, :] = ybs * np.cos(angle)
                zbs_surf[i, :] = ybs * np.sin(angle)

            fig.add_trace(go.Surface(
                x=xbs_surf, y=ybs_surf, z=zbs_surf,
                colorscale=[[0, '#32a852'], [1, '#32a852']],
                showscale=False,
                name='Bow shock',
                opacity=0.2,
                hoverinfo='skip'
            ))

            # Add field lines
            for i, (X, Y, Z) in enumerate(field_lines):
                fig.add_trace(go.Scatter3d(
                    x=X, y=Y, z=Z,
                    mode='lines',
                    name='Field lines' if i == 0 else '',
                    line=dict(color='#1f77b4', width=3),
                    opacity=0.8,
                    showlegend=(i == 0),
                    hoverinfo='skip'
                ))

            # Add full Wind trajectory
            fig.add_trace(go.Scatter3d(
                x=xyz_w[:, 0], y=xyz_w[:, 1], z=xyz_w[:, 2],
                mode='lines',
                name='Wind trajectory',
                line=dict(color='black', width=4),
                opacity=0.8,
                hoverinfo='skip'
            ))

            # Add broad time window
            has_broad = False
            for c in crossings:
                if c.has_broad_window():
                    broad_indices = [i for i, t in enumerate(times_w) if c.broad_start <= t <= c.broad_end]
                    if broad_indices:
                        idx_start, idx_end = broad_indices[0], broad_indices[-1]
                        fig.add_trace(go.Scatter3d(
                            x=xyz_w[idx_start:idx_end+1, 0],
                            y=xyz_w[idx_start:idx_end+1, 1],
                            z=xyz_w[idx_start:idx_end+1, 2],
                            mode='lines',
                            name='Broad window' if not has_broad else '',
                            line=dict(color='orange', width=5),
                            opacity=0.6,
                            showlegend=not has_broad,
                            hoverinfo='skip'
                        ))
                        has_broad = True

            # Add zoom time window
            has_zoom = False
            for c in crossings:
                if c.has_zoom_window():
                    zoom_indices = [i for i, t in enumerate(times_w) if c.zoom_start <= t <= c.zoom_end]
                    if zoom_indices:
                        idx_start, idx_end = zoom_indices[0], zoom_indices[-1]
                        fig.add_trace(go.Scatter3d(
                            x=xyz_w[idx_start:idx_end+1, 0],
                            y=xyz_w[idx_start:idx_end+1, 1],
                            z=xyz_w[idx_start:idx_end+1, 2],
                            mode='lines',
                            name='Zoom window' if not has_zoom else '',
                            line=dict(color='red', width=5),
                            opacity=0.8,
                            showlegend=not has_zoom,
                            hoverinfo='skip'
                        ))
                        has_zoom = True

            # Add crossings as scatter points
            crossing_color = 'red' if direction == 'in' else 'darkblue'
            crossing_symbol = 'diamond' if direction == 'in' else 'diamond-open'

            crossing_x, crossing_y, crossing_z = [], [], []
            crossing_times = []

            for c in crossings:
                time_diffs = [abs((t - c.time).total_seconds()) for t in times_w]
                if time_diffs:
                    idx = np.argmin(time_diffs)
                    crossing_x.append(xyz_w[idx, 0])
                    crossing_y.append(xyz_w[idx, 1])
                    crossing_z.append(xyz_w[idx, 2])
                    crossing_times.append(c.time.strftime('%Y-%m-%d %H:%M:%S'))

            fig.add_trace(go.Scatter3d(
                x=crossing_x, y=crossing_y, z=crossing_z,
                mode='markers',
                name=f'{direction_name} crossings',
                marker=dict(
                    size=8,
                    color=crossing_color,
                    symbol=crossing_symbol,
                    line=dict(color='black', width=2),
                    opacity=0.9
                ),
                text=crossing_times,
                hovertemplate='<b>%{text}</b><br>X=%{x:.2f} Re<br>Y=%{y:.2f} Re<br>Z=%{z:.2f} Re<extra></extra>'
            ))

            # Add Earth sphere with better appearance
            u = np.linspace(0, 2 * np.pi, 50)
            v = np.linspace(0, np.pi, 30)
            x_earth = np.outer(np.cos(u), np.sin(v))
            y_earth = np.outer(np.sin(u), np.sin(v))
            z_earth = np.outer(np.ones(np.size(u)), np.cos(v))

            # Earth surface (blue ocean with green continents)
            fig.add_trace(go.Surface(
                x=x_earth, y=y_earth, z=z_earth,
                colorscale=[[0, '#1a4d7a'], [0.5, '#2e7cb8'], [1, '#2e7cb8']],
                showscale=False,
                name='Earth',
                hoverinfo='skip',
                lighting=dict(ambient=0.6, diffuse=0.8, specular=0.3)
            ))

            # Add latitude gridlines
            for lat in np.linspace(-np.pi/2, np.pi/2, 7):
                lons = np.linspace(0, 2*np.pi, 100)
                x_lat = np.cos(lat) * np.cos(lons)
                y_lat = np.cos(lat) * np.sin(lons)
                z_lat = np.sin(lat) * np.ones_like(lons)
                fig.add_trace(go.Scatter3d(
                    x=x_lat, y=y_lat, z=z_lat,
                    mode='lines',
                    line=dict(color='rgba(255,255,255,0.3)', width=1),
                    showlegend=False,
                    hoverinfo='skip'
                ))

            # Add longitude gridlines
            for lon in np.linspace(0, 2*np.pi, 12, endpoint=False):
                lats = np.linspace(-np.pi/2, np.pi/2, 100)
                x_lon = np.cos(lats) * np.cos(lon)
                y_lon = np.cos(lats) * np.sin(lon)
                z_lon = np.sin(lats)
                fig.add_trace(go.Scatter3d(
                    x=x_lon, y=y_lon, z=z_lon,
                    mode='lines',
                    line=dict(color='rgba(255,255,255,0.3)', width=1),
                    showlegend=False,
                    hoverinfo='skip'
                ))

            # Update layout
            fig.update_layout(
                title=dict(text=""),
                scene=dict(
                    xaxis_title="X<sub>GSM</sub> [R<sub>⊕</sub>]",
                    yaxis_title="Y<sub>GSM</sub> [R<sub>⊕</sub>]",
                    zaxis_title="Z<sub>GSM</sub> [R<sub>⊕</sub>]",
                    xaxis=dict(range=[-40, 40]),
                    yaxis=dict(range=[-40, 40]),
                    zaxis=dict(range=[-40, 40]),
                ),
                width=1200,
                height=800,
                hovermode='closest'
            )

            # Build crossing information for HTML header
            crossing_info = f"📍 Pass {pass_id} - {direction_name} Crossings: {len(crossings)}\n\n"

            for c in crossings:
                if c.has_broad_window():
                    crossing_info += f"     Broad window:  {c.broad_start.strftime('%Y-%m-%d %H:%M:%S')} to {c.broad_end.strftime('%Y-%m-%d %H:%M:%S')}\n"
                    break

            for c in crossings:
                if c.has_zoom_window():
                    crossing_info += f"     Zoom window:   {c.zoom_start.strftime('%Y-%m-%d %H:%M:%S')} to {c.zoom_end.strftime('%Y-%m-%d %H:%M:%S')}\n"
                    break

            if crossings:
                crossing_info += f"     Time range: {min(c.time for c in crossings).strftime('%Y-%m-%d %H:%M:%S')} to {max(c.time for c in crossings).strftime('%Y-%m-%d %H:%M:%S')}\n"

            crossing_info += "-" * 80 + "\n"
            for i, c in enumerate(crossings, 1):
                crossing_info += f"  {i}. Crossing: {c.time.strftime('%Y-%m-%d %H:%M:%S')}\n"

            # Save the figure as HTML with crossing info
            import os
            from pathlib import Path

            # Create figs directory if it doesn't exist
            figs_dir = Path("figs")
            figs_dir.mkdir(exist_ok=True)

            html_file = figs_dir / f"Magnetosphere_Pass_{pass_id:02d}_{direction_name}_{year:04d}-{month:02d}-{day:02d}T{hour:02d}{minute:02d}.html"

            # Create HTML with crossing info and embedded plot
            plot_html = fig.to_html(include_plotlyjs='cdn')

            full_html = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Magnetosphere Pass {pass_id} - {direction_name} Crossings</title>
    <style>
        body {{
            font-family: monospace;
            margin: 20px;
            background-color: #f5f5f5;
        }}
        .header {{
            background-color: white;
            padding: 20px;
            border-radius: 5px;
            margin-bottom: 20px;
            white-space: pre;
            line-height: 1.6;
        }}
        .plot-container {{
            background-color: white;
            padding: 20px;
            border-radius: 5px;
        }}
    </style>
</head>
<body>
    <div class="header">{crossing_info}</div>
    <div class="plot-container">
        {plot_html}
    </div>
</body>
</html>"""

            with open(str(html_file), 'w', encoding='utf-8') as f:
                f.write(full_html)

            abs_path = html_file.absolute()
            print(f"\n   📄 Interactive plot saved!")
            print(f"   📍 File: {html_file}")
            print(f"   📂 Full path: {abs_path}")
            print(f"\n   ✅ Open this file in your web browser to view the interactive 3D plot")
            print("\n" + "="*80)
            print("✅ 3D Visualization complete!")
            print("   💡 Tip: Use mouse to rotate (click & drag)")
            print("      Scroll to zoom, right-click to pan")

        except Exception as e:
            print(f"❌ Error: {type(e).__name__}: {e}")
            import traceback
            traceback.print_exc()

# Attach event handlers
pass_dropdown.observe(on_selection_changed, names='value')
direction_dropdown.observe(on_selection_changed, names='value')
run_button.on_click(on_run_clicked)

# Trigger initial update
on_selection_changed({'new': all_passes[0]})

# ============================================================================
# Display
# ============================================================================

print("="*80)
print("PASS + DIRECTION MAGNETOSPHERE VIEWER (3D - INTERACTIVE)")
print("="*80 + "\n")

display(widgets.HTML("<h3>📍 Select Pass and Direction:</h3>"))
display(pass_dropdown)
display(direction_dropdown)
display(run_button)

display(widgets.HTML("<hr>"))
display(widgets.HTML("<h3>📋 Crossings:</h3>"))
display(info_output)

display(widgets.HTML("<hr>"))
display(widgets.HTML("<h3>🎲 Interactive 3D Magnetosphere:</h3>"))
display(viz_output)

print("\n✅ Select pass and direction, then click the button to generate the interactive 3D visualization.")
print("💡 Use MOUSE:")
print("   • Click & drag to ROTATE")
print("   • Scroll to ZOOM")
print("   • Right-click & drag to PAN")

Loading bow shock crossings database...

✓ Loaded 631 crossings in 69 passes

PASS + DIRECTION MAGNETOSPHERE VIEWER (3D - INTERACTIVE)



HTML(value='<h3>📍 Select Pass and Direction:</h3>')

Dropdown(description='Pass ID:', options=(('Pass 2', 2), ('Pass 3', 3), ('Pass 4', 4), ('Pass 5', 5), ('Pass 6…

Dropdown(description='Direction:', options=(('Inbound (entering bow shock)', 'in'), ('Outbound (leaving bow sh…

Button(button_style='info', description='Generate Interactive 3D Visualization', icon='cube', style=ButtonStyl…

...

Output()

HTML(value='<hr>')

HTML(value='<h3>🎲 Interactive 3D Magnetosphere:</h3>')

Output()


✅ Select pass and direction, then click the button to generate the interactive 3D visualization.
💡 Use MOUSE:
   • Click & drag to ROTATE
   • Scroll to ZOOM
   • Right-click & drag to PAN


## BATCH PROCESSING CELL - Generate 3D visualizations for all passes

In [ ]:
"""
BATCH PROCESSING CELL - Generate 3D visualizations for all passes

Runs the magnetosphere visualization for all passes (both inbound and outbound).
Handles failures gracefully and reports results at the end.

Run this cell overnight to generate all HTML files.
"""

print("=" * 80)
print("BATCH PROCESSING: Generating 3D Magnetosphere Visualizations for All Passes")
print("=" * 80)
print()

import time
from datetime import datetime

# Get all passes
all_passes = db.get_passes()
print(f"Found {len(all_passes)} passes to process\n")

# Track results
results = {
    'success': [],
    'failed': []
}

start_time = datetime.now()

# Process each pass
for pass_idx, pass_id in enumerate(all_passes, 1):
    print(f"[{pass_idx}/{len(all_passes)}] Processing Pass {pass_id}...\n")

    # Get crossings for this pass
    all_crossings = db.get_crossings_for_pass(pass_id)

    # Try both directions
    for direction, direction_name in [('in', 'Inbound'), ('out', 'Outbound')]:
        crossings = [c for c in all_crossings if c.crossing_direction == direction]

        if not crossings:
            print(f"     ⊘ No {direction_name.lower()} crossings - skipping")
            continue

        try:
            print(f"     Processing {direction_name}...", end=" ")

            # Find reference time
            t0 = None
            for c in crossings:
                if c.has_zoom_window():
                    t0 = c.zoom_start + (c.zoom_end - c.zoom_start) / 2
                    break

            if t0 is None:
                t0 = crossings[0].time

            year, month, day = t0.year, t0.month, t0.day
            hour, minute = t0.hour, t0.minute
            t_iso = f"{year:04d}-{month:02d}-{day:02d}T{hour:02d}:{minute:02d}:00"

            # Fetch OMNI2 data with progressive time window expansion
            Pdyn, Dst, By, Bz, Ma = None, None, None, None, None
            time_windows = [
                (0, "exact timestamp"),
                (10, "±10 minutes"),
                (30, "±30 minutes"),
                (60, "±1 hour"),
                (120, "±2 hours"),
                (240, "±4 hours"),
            ]

            for window_minutes, window_desc in time_windows:
                try:
                    if window_minutes == 0:
                        Pdyn, Dst, By, Bz, Ma = get_omni2_params(t_iso)
                    else:
                        t_start = t0 - dt.timedelta(minutes=window_minutes)
                        t_end = t0 + dt.timedelta(minutes=window_minutes)
                        values = []
                        for t_fetch in [t_start, t0, t_end]:
                            try:
                                vals = get_omni2_params(t_fetch.isoformat())
                                if all(np.isfinite(v) for v in vals):
                                    values.append(vals)
                            except:
                                pass

                        if not values:
                            raise ValueError(f"No valid data")

                        Pdyn = np.mean([v[0] for v in values])
                        Dst = np.mean([v[1] for v in values])
                        By = np.mean([v[2] for v in values])
                        Bz = np.mean([v[3] for v in values])
                        Ma = np.mean([v[4] for v in values])

                    if np.isfinite(Pdyn) and np.isfinite(Bz):
                        break
                except:
                    pass

            if not (np.isfinite(Pdyn) and np.isfinite(Bz)):
                raise Exception("Could not fetch valid OMNI2 data")

            # Compute boundaries
            parmod = t96_parmod(Pdyn, Dst, By, Bz)
            xmp, ymp, r0mp, alphamp = shue98_magnetopause_rtheta(Pdyn, Bz, ntheta=361)
            xbs, ybs, rbs0, delta_bs = bowshock_rtheta_from_Ma(r0mp, Ma, ntheta=361)

            # Trace field lines
            seeds = []
            r_seed = 2.5
            for lat in np.linspace(-np.pi/2, np.pi/2, 6):
                for lon in np.linspace(0, 2*np.pi, 12, endpoint=False):
                    x = r_seed * np.cos(lat) * np.cos(lon)
                    y = r_seed * np.cos(lat) * np.sin(lon)
                    z = r_seed * np.sin(lat)
                    seeds.append((x, y, z))

            for r in [3.5, 5.0]:
                for angle in np.linspace(0, np.pi/2, 6):
                    for sign in [1, -1]:
                        seeds.append((-r * np.cos(angle), 0, sign * r * np.sin(angle)))

            field_lines = []
            ut = unix_seconds(t0)
            for s in seeds:
                X, Y, Z, ps = trace_field_line(s, parmod, ut, rlim=60.0, r0=1.0)
                field_lines.append((X, Y, Z))

            # Fetch Wind
            start = (t0 - dt.timedelta(hours=12)).isoformat()
            stop = (t0 + dt.timedelta(hours=12)).isoformat()
            times_w, xyz_w = get_wind_gsm_from_ssc(start, stop, observatory_id="wind")

            # Create Plotly figure
            fig = go.Figure()

            # Add magnetopause
            angles_mp = np.linspace(0, 2*np.pi, 100)
            xmp_flat, ymp_flat, zmp_flat = [], [], []
            for angle in angles_mp:
                xmp_flat.extend(xmp)
                ymp_flat.extend(ymp * np.cos(angle))
                zmp_flat.extend(ymp * np.sin(angle))

            faces_i, faces_j, faces_k = [], [], []
            n_angles, n_radii = len(angles_mp), len(xmp)
            for i in range(n_angles - 1):
                for j in range(n_radii - 1):
                    faces_i.extend([i * n_radii + j, (i + 1) * n_radii + j])
                    faces_j.extend([(i + 1) * n_radii + j, (i + 1) * n_radii + (j + 1)])
                    faces_k.extend([i * n_radii + (j + 1), i * n_radii + (j + 1)])

            fig.add_trace(go.Mesh3d(
                x=xmp_flat, y=ymp_flat, z=zmp_flat,
                i=faces_i, j=faces_j, k=faces_k,
                color='#326ba8', name='Magnetopause', opacity=0.35, showlegend=True, hoverinfo='skip'
            ))

            # Add bow shock
            angles_bs = np.linspace(0, 2*np.pi, 80)
            xbs_surf = np.tile(xbs, (len(angles_bs), 1))
            ybs_surf = np.zeros((len(angles_bs), len(xbs)))
            zbs_surf = np.zeros((len(angles_bs), len(xbs)))
            for i, angle in enumerate(angles_bs):
                ybs_surf[i, :] = ybs * np.cos(angle)
                zbs_surf[i, :] = ybs * np.sin(angle)

            fig.add_trace(go.Surface(
                x=xbs_surf, y=ybs_surf, z=zbs_surf,
                colorscale=[[0, '#32a852'], [1, '#32a852']], showscale=False,
                name='Bow shock', opacity=0.2, hoverinfo='skip'
            ))

            # Add field lines
            for i, (X, Y, Z) in enumerate(field_lines):
                fig.add_trace(go.Scatter3d(
                    x=X, y=Y, z=Z, mode='lines', name='Field lines' if i == 0 else '',
                    line=dict(color='#1f77b4', width=3), opacity=0.8,
                    showlegend=(i == 0), hoverinfo='skip'
                ))

            # Add Wind trajectory
            fig.add_trace(go.Scatter3d(
                x=xyz_w[:, 0], y=xyz_w[:, 1], z=xyz_w[:, 2], mode='lines',
                name='Wind trajectory', line=dict(color='black', width=4), opacity=0.8, hoverinfo='skip'
            ))

            # Add time windows
            has_broad = False
            for c in crossings:
                if c.has_broad_window():
                    broad_indices = [i for i, t in enumerate(times_w) if c.broad_start <= t <= c.broad_end]
                    if broad_indices:
                        idx_start, idx_end = broad_indices[0], broad_indices[-1]
                        fig.add_trace(go.Scatter3d(
                            x=xyz_w[idx_start:idx_end+1, 0], y=xyz_w[idx_start:idx_end+1, 1], z=xyz_w[idx_start:idx_end+1, 2],
                            mode='lines', name='Broad window' if not has_broad else '',
                            line=dict(color='orange', width=5), opacity=0.6,
                            showlegend=not has_broad, hoverinfo='skip'
                        ))
                        has_broad = True

            has_zoom = False
            for c in crossings:
                if c.has_zoom_window():
                    zoom_indices = [i for i, t in enumerate(times_w) if c.zoom_start <= t <= c.zoom_end]
                    if zoom_indices:
                        idx_start, idx_end = zoom_indices[0], zoom_indices[-1]
                        fig.add_trace(go.Scatter3d(
                            x=xyz_w[idx_start:idx_end+1, 0], y=xyz_w[idx_start:idx_end+1, 1], z=xyz_w[idx_start:idx_end+1, 2],
                            mode='lines', name='Zoom window' if not has_zoom else '',
                            line=dict(color='red', width=5), opacity=0.8,
                            showlegend=not has_zoom, hoverinfo='skip'
                        ))
                        has_zoom = True

            # Add crossings
            crossing_color = 'red' if direction == 'in' else 'darkblue'
            crossing_symbol = 'diamond' if direction == 'in' else 'diamond-open'
            crossing_x, crossing_y, crossing_z, crossing_times = [], [], [], []
            for c in crossings:
                time_diffs = [abs((t - c.time).total_seconds()) for t in times_w]
                if time_diffs:
                    idx = np.argmin(time_diffs)
                    crossing_x.append(xyz_w[idx, 0])
                    crossing_y.append(xyz_w[idx, 1])
                    crossing_z.append(xyz_w[idx, 2])
                    crossing_times.append(c.time.strftime('%Y-%m-%d %H:%M:%S'))

            fig.add_trace(go.Scatter3d(
                x=crossing_x, y=crossing_y, z=crossing_z, mode='markers',
                name=f'{direction_name} crossings',
                marker=dict(size=8, color=crossing_color, symbol=crossing_symbol,
                           line=dict(color='black', width=2), opacity=0.9),
                text=crossing_times,
                hovertemplate='<b>%{text}</b><br>X=%{x:.2f} Re<br>Y=%{y:.2f} Re<br>Z=%{z:.2f} Re<extra></extra>'
            ))

            # Add Earth
            u = np.linspace(0, 2 * np.pi, 50)
            v = np.linspace(0, np.pi, 30)
            x_earth = np.outer(np.cos(u), np.sin(v))
            y_earth = np.outer(np.sin(u), np.sin(v))
            z_earth = np.outer(np.ones(np.size(u)), np.cos(v))

            fig.add_trace(go.Surface(
                x=x_earth, y=y_earth, z=z_earth,
                colorscale=[[0, '#1a4d7a'], [0.5, '#2e7cb8'], [1, '#2e7cb8']],
                showscale=False, name='Earth', hoverinfo='skip',
                lighting=dict(ambient=0.6, diffuse=0.8, specular=0.3)
            ))

            # Add latitude/longitude gridlines
            for lat in np.linspace(-np.pi/2, np.pi/2, 7):
                lons = np.linspace(0, 2*np.pi, 100)
                x_lat = np.cos(lat) * np.cos(lons)
                y_lat = np.cos(lat) * np.sin(lons)
                z_lat = np.sin(lat) * np.ones_like(lons)
                fig.add_trace(go.Scatter3d(
                    x=x_lat, y=y_lat, z=z_lat, mode='lines',
                    line=dict(color='rgba(255,255,255,0.3)', width=1),
                    showlegend=False, hoverinfo='skip'
                ))

            for lon in np.linspace(0, 2*np.pi, 12, endpoint=False):
                lats = np.linspace(-np.pi/2, np.pi/2, 100)
                x_lon = np.cos(lats) * np.cos(lon)
                y_lon = np.cos(lats) * np.sin(lon)
                z_lon = np.sin(lats)
                fig.add_trace(go.Scatter3d(
                    x=x_lon, y=y_lon, z=z_lon, mode='lines',
                    line=dict(color='rgba(255,255,255,0.3)', width=1),
                    showlegend=False, hoverinfo='skip'
                ))

            # Update layout
            fig.update_layout(
                title=dict(text=""),
                scene=dict(
                    xaxis_title="X<sub>GSM</sub> [R<sub>⊕</sub>]",
                    yaxis_title="Y<sub>GSM</sub> [R<sub>⊕</sub>]",
                    zaxis_title="Z<sub>GSM</sub> [R<sub>⊕</sub>]",
                    xaxis=dict(range=[-40, 40]),
                    yaxis=dict(range=[-40, 40]),
                    zaxis=dict(range=[-40, 40]),
                ),
                width=1200, height=800, hovermode='closest'
            )

            # Build crossing info for HTML
            crossing_info = f"📍 Pass {pass_id} - {direction_name} Crossings: {len(crossings)}\n\n"
            for c in crossings:
                if c.has_broad_window():
                    crossing_info += f"     Broad window:  {c.broad_start.strftime('%Y-%m-%d %H:%M:%S')} to {c.broad_end.strftime('%Y-%m-%d %H:%M:%S')}\n"
                    break
            for c in crossings:
                if c.has_zoom_window():
                    crossing_info += f"     Zoom window:   {c.zoom_start.strftime('%Y-%m-%d %H:%M:%S')} to {c.zoom_end.strftime('%Y-%m-%d %H:%M:%S')}\n"
                    break
            if crossings:
                crossing_info += f"     Time range: {min(c.time for c in crossings).strftime('%Y-%m-%d %H:%M:%S')} to {max(c.time for c in crossings).strftime('%Y-%m-%d %H:%M:%S')}\n"
            crossing_info += "-" * 80 + "\n"
            for i, c in enumerate(crossings, 1):
                crossing_info += f"  {i}. Crossing: {c.time.strftime('%Y-%m-%d %H:%M:%S')}\n"

            # Save HTML
            import os
            from pathlib import Path
            figs_dir = Path("figs")
            figs_dir.mkdir(exist_ok=True)
            html_file = figs_dir / f"Magnetosphere_Pass_{pass_id:02d}_{direction_name}_{year:04d}-{month:02d}-{day:02d}T{hour:02d}{minute:02d}.html"
            plot_html = fig.to_html(include_plotlyjs='cdn')
            full_html = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Magnetosphere Pass {pass_id} - {direction_name} Crossings</title>
    <style>
        body {{font-family: monospace; margin: 20px; background-color: #f5f5f5;}}
        .header {{background-color: white; padding: 20px; border-radius: 5px; margin-bottom: 20px; white-space: pre; line-height: 1.6;}}
        .plot-container {{background-color: white; padding: 20px; border-radius: 5px;}}
    </style>
</head>
<body>
    <div class="header">{crossing_info}</div>
    <div class="plot-container">{plot_html}</div>
</body>
</html>"""

            with open(str(html_file), 'w', encoding='utf-8') as f:
                f.write(full_html)

            print(f"✓ Saved")
            results['success'].append(f"Pass {pass_id} ({direction_name})")

        except Exception as e:
            print(f"✗ ERROR: {type(e).__name__}: {str(e)[:60]}")
            results['failed'].append({
                'pass': pass_id,
                'direction': direction_name,
                'error': f"{type(e).__name__}: {str(e)[:80]}"
            })

    print()

# Summary
end_time = datetime.now()
elapsed = end_time - start_time

print("=" * 80)
print("BATCH PROCESSING COMPLETE")
print("=" * 80)
print(f"\n⏱️  Time elapsed: {elapsed}")
print(f"\n✅ Successful: {len(results['success'])}")
for item in results['success']:
    print(f"   • {item}")

if results['failed']:
    print(f"\n❌ Failed: {len(results['failed'])}")
    for item in results['failed']:
        print(f"   • Pass {item['pass']} ({item['direction']})")
        print(f"     → {item['error']}")
else:
    print(f"\n🎉 No failures!")

print("\n" + "=" * 80)
print(f"All HTML files saved to: figs/")
print("=" * 80)

## updated range 

In [ ]:
"""
BATCH PROCESSING: 18 Failed Passes with Wind Data (CORRECTED)
Uses proper Plotly visualization from working OMNI code
Generates 3D magnetosphere plots with dynamic range adjustment
"""

print("=" * 80)
print("BATCH PROCESSING: 18 FAILED PASSES WITH WIND DATA (CORRECTED)")
print("=" * 80)
print()

from datetime import datetime, timedelta
from pathlib import Path
import os
import pandas as pd
import numpy as np
import datetime as dt
import plotly.graph_objects as go

# List of 18 failed passes to process
failed_passes = [
    (3, 'out', 1995, 9, 16, 10, 17),
    (4, 'in', 1995, 11, 28, 8, 40),
    (5, 'in', 1995, 12, 20, 21, 40),
    (5, 'out', 1995, 12, 22, 13, 46),
    (6, 'out', 1996, 1, 14, 7, 10),
    (8, 'in', 1996, 4, 17, 14, 15),
    (9, 'in', 1996, 5, 9, 14, 0),
    (13, 'out', 1996, 11, 16, 19, 0),
    (15, 'out', 1997, 1, 1, 16, 30),
    (16, 'out', 1997, 6, 12, 13, 55),
    (19, 'out', 1997, 9, 9, 19, 21),
    (20, 'out', 1997, 10, 1, 12, 15),
    (22, 'in', 1998, 7, 1, 18, 18),
    (24, 'out', 1998, 8, 16, 22, 37),
    (31, 'in', 1999, 2, 4, 11, 27),
    (42, 'out', 1999, 11, 18, 6, 45),
    (55, 'in', 2001, 2, 20, 2, 45),
    (56, 'in', 2001, 9, 8, 12, 5),
]

# TEMPORARY: Only process 3 passes for testing
failed_passes = [
    (42, 'out', 1999, 11, 18, 6, 45),
    (55, 'in', 2001, 2, 20, 2, 45),
    (56, 'in', 2001, 9, 8, 12, 5),
]

print(f"Found {len(failed_passes)} failed passes to process\n")

results = {'success': [], 'failed': []}
start_time = datetime.now()

# Process each failed pass
for pass_idx, (pass_id, direction, year, month, day, hour, minute) in enumerate(failed_passes, 1):
    direction_name = "Inbound" if direction == 'in' else "Outbound"
    print(f"[{pass_idx:2d}/{len(failed_passes)}] Pass {pass_id:2d} ({direction_name:7s}) {year}-{month:02d}-{day:02d}T{hour:02d}:{minute:02d}...")

    # Get crossings for this pass from database
    all_crossings = db.get_crossings_for_pass(pass_id)
    crossings = [c for c in all_crossings if c.crossing_direction == direction]

    if not crossings:
        print(f"     ⊘ No {direction_name.lower()} crossings - skipping\n")
        continue

    try:
        # Find reference time for visualization
        t0 = None
        for c in crossings:
            if c.has_zoom_window():
                t0 = c.zoom_start + (c.zoom_end - c.zoom_start) / 2
                break
        if t0 is None:
            t0 = crossings[0].time

        print(f"     Extracting Wind parameters...", end=" ")

        # Extract Wind parameters with extended time tolerances (±6 hours for data gaps)
        params = get_wind_params(year, month, day, hour, minute,
                                mfi_tolerance=21600,  # ±6 hours instead of ±5 min
                                swe_tolerance=21600)   # ±6 hours instead of ±30 min

        date_used = f"{year}-{month:02d}-{day:02d}"

        if params is None:
            raise Exception("Could not extract Wind parameters (even with ±6h tolerance)")

        Pdyn = params['Pdyn']
        Dst = params['Dst']
        By = params['By']
        Bz = params['Bz']
        Ma = params['Ma']

        if not (np.isfinite(Pdyn) and np.isfinite(Bz)):
            raise Exception(f"Invalid params: Pdyn={Pdyn}, Bz={Bz}")

        print(f"✓")
        print(f"     Parameters: Pdyn={Pdyn:.2f}nPa, Bz={Bz:.2f}nT, Ma={Ma:.1f}, Dst={Dst:.0f}nT")

        # Compute magnetospheric boundaries
        print(f"     Computing boundaries...", end=" ")
        parmod = t96_parmod(Pdyn, Dst, By, Bz)
        xmp, ymp, r0mp, alphamp = shue98_magnetopause_rtheta(Pdyn, Bz, ntheta=361)
        xbs, ybs, rbs0, delta_bs = bowshock_rtheta_from_Ma(r0mp, Ma, ntheta=361)
        print(f"✓")

        # Verify boundaries are valid
        if not (np.isfinite(xmp).all() and np.isfinite(ymp).all()):
            raise Exception("Magnetopause calculation failed")
        if not (np.isfinite(xbs).all() and np.isfinite(ybs).all()):
            raise Exception("Bow shock calculation failed")

        # Trace field lines
        print(f"     Tracing field lines...", end=" ")
        seeds = []
        r_seed = 2.5
        for lat in np.linspace(-np.pi/2, np.pi/2, 6):
            for lon in np.linspace(0, 2*np.pi, 12, endpoint=False):
                x = r_seed * np.cos(lat) * np.cos(lon)
                y = r_seed * np.cos(lat) * np.sin(lon)
                z = r_seed * np.sin(lat)
                seeds.append((x, y, z))

        for r in [3.5, 5.0]:
            for angle in np.linspace(0, np.pi/2, 6):
                for sign in [1, -1]:
                    seeds.append((-r * np.cos(angle), 0, sign * r * np.sin(angle)))

        field_lines = []
        ut = unix_seconds(t0)
        for s in seeds:
            try:
                result = trace_field_line(s, parmod, ut, rlim=60.0, r0=1.0)
                if result is not None:
                    X, Y, Z, ps = result
                    field_lines.append((X, Y, Z))
            except:
                pass  # Skip seeds that fail to trace

        print(f"✓ ({len(field_lines)} lines)")

        # Fetch Wind trajectory
        print(f"     Fetching Wind trajectory...", end=" ")
        start = (t0 - dt.timedelta(hours=12)).isoformat()
        stop = (t0 + dt.timedelta(hours=12)).isoformat()
        times_w, xyz_w = get_wind_gsm_from_ssc(start, stop, observatory_id="wind")
        print(f"✓ ({len(times_w)} points)")

        # Calculate plot range based on Wind trajectory
        print(f"     Calculating plot range...", end=" ")
        if len(xyz_w) > 0:
            x_max_dist = max(abs(xyz_w[:, 0].min()), abs(xyz_w[:, 0].max()))
            y_max_dist = max(abs(xyz_w[:, 1].min()), abs(xyz_w[:, 1].max()))
            z_max_dist = max(abs(xyz_w[:, 2].min()), abs(xyz_w[:, 2].max()))

            # Ensure minimum range and add 20% padding
            x_max_dist = max(x_max_dist, 5) * 1.2
            y_max_dist = max(y_max_dist, 5) * 1.2
            z_max_dist = max(z_max_dist, 5) * 1.2

            x_lim = [-x_max_dist, x_max_dist]
            y_lim = [-y_max_dist, y_max_dist]
            z_lim = [-z_max_dist, z_max_dist]
        else:
            x_lim = y_lim = z_lim = [-40, 40]

        print(f"✓ X=[{x_lim[0]:.1f}, {x_lim[1]:.1f}]")

        # Create Plotly figure
        print(f"     Building visualization...", end=" ")
        fig = go.Figure()

        # ===== ADD MAGNETOPAUSE =====
        angles_mp = np.linspace(0, 2*np.pi, 100)
        xmp_flat, ymp_flat, zmp_flat = [], [], []
        for angle in angles_mp:
            xmp_flat.extend(xmp)
            ymp_flat.extend(ymp * np.cos(angle))
            zmp_flat.extend(ymp * np.sin(angle))

        faces_i, faces_j, faces_k = [], [], []
        n_angles, n_radii = len(angles_mp), len(xmp)
        for i in range(n_angles - 1):
            for j in range(n_radii - 1):
                v0 = i * n_radii + j
                v1 = i * n_radii + (j + 1)
                v2 = ((i + 1) % n_angles) * n_radii + j
                v3 = ((i + 1) % n_angles) * n_radii + (j + 1)

                faces_i.extend([v0, v0, v2])
                faces_j.extend([v1, v2, v3])
                faces_k.extend([v2, v3, v1])

        fig.add_trace(go.Mesh3d(
            x=xmp_flat, y=ymp_flat, z=zmp_flat,
            i=faces_i, j=faces_j, k=faces_k,
            color='#326ba8', name='Magnetopause', opacity=0.35, showlegend=True, hoverinfo='skip'
        ))

        # ===== ADD BOW SHOCK =====
        angles_bs = np.linspace(0, 2*np.pi, 80)
        xbs_surf = np.tile(xbs, (len(angles_bs), 1))
        ybs_surf = np.zeros((len(angles_bs), len(xbs)))
        zbs_surf = np.zeros((len(angles_bs), len(xbs)))
        for i, angle in enumerate(angles_bs):
            ybs_surf[i, :] = ybs * np.cos(angle)
            zbs_surf[i, :] = ybs * np.sin(angle)

        fig.add_trace(go.Surface(
            x=xbs_surf, y=ybs_surf, z=zbs_surf,
            colorscale=[[0, '#32a852'], [1, '#32a852']], showscale=False,
            name='Bow shock', opacity=0.2, hoverinfo='skip'
        ))

        # ===== ADD FIELD LINES =====
        for i, (X, Y, Z) in enumerate(field_lines):
            fig.add_trace(go.Scatter3d(
                x=X, y=Y, z=Z, mode='lines', name='Field lines' if i == 0 else '',
                line=dict(color='#1f77b4', width=3), opacity=0.8,
                showlegend=(i == 0), hoverinfo='skip'
            ))

        # ===== ADD WIND TRAJECTORY =====
        fig.add_trace(go.Scatter3d(
            x=xyz_w[:, 0], y=xyz_w[:, 1], z=xyz_w[:, 2], mode='lines',
            name='Wind trajectory', line=dict(color='black', width=4), opacity=0.8, hoverinfo='skip'
        ))

        # ===== ADD TIME WINDOWS =====
        has_broad = False
        for c in crossings:
            if c.has_broad_window():
                broad_indices = [i for i, t in enumerate(times_w) if c.broad_start <= t <= c.broad_end]
                if broad_indices:
                    idx_start, idx_end = broad_indices[0], broad_indices[-1]
                    fig.add_trace(go.Scatter3d(
                        x=xyz_w[idx_start:idx_end+1, 0], y=xyz_w[idx_start:idx_end+1, 1], z=xyz_w[idx_start:idx_end+1, 2],
                        mode='lines', name='Broad window' if not has_broad else '',
                        line=dict(color='orange', width=5), opacity=0.6,
                        showlegend=not has_broad, hoverinfo='skip'
                    ))
                    has_broad = True

        has_zoom = False
        for c in crossings:
            if c.has_zoom_window():
                zoom_indices = [i for i, t in enumerate(times_w) if c.zoom_start <= t <= c.zoom_end]
                if zoom_indices:
                    idx_start, idx_end = zoom_indices[0], zoom_indices[-1]
                    fig.add_trace(go.Scatter3d(
                        x=xyz_w[idx_start:idx_end+1, 0], y=xyz_w[idx_start:idx_end+1, 1], z=xyz_w[idx_start:idx_end+1, 2],
                        mode='lines', name='Zoom window' if not has_zoom else '',
                        line=dict(color='red', width=5), opacity=0.8,
                        showlegend=not has_zoom, hoverinfo='skip'
                    ))
                    has_zoom = True

        # ===== ADD CROSSING MARKERS =====
        crossing_color = 'red' if direction == 'in' else 'darkblue'
        crossing_symbol = 'diamond' if direction == 'in' else 'diamond-open'
        crossing_x, crossing_y, crossing_z, crossing_times = [], [], [], []

        for c in crossings:
            time_diffs = [abs((t - c.time).total_seconds()) for t in times_w]
            if time_diffs:
                idx = np.argmin(time_diffs)
                crossing_x.append(xyz_w[idx, 0])
                crossing_y.append(xyz_w[idx, 1])
                crossing_z.append(xyz_w[idx, 2])
                crossing_times.append(c.time.strftime('%Y-%m-%d %H:%M:%S'))

        fig.add_trace(go.Scatter3d(
            x=crossing_x, y=crossing_y, z=crossing_z, mode='markers',
            name=f'{direction_name} crossings',
            marker=dict(size=8, color=crossing_color, symbol=crossing_symbol,
                       line=dict(color='black', width=2), opacity=0.9),
            text=crossing_times,
            hovertemplate='<b>%{text}</b><br>X=%{x:.2f} Re<br>Y=%{y:.2f} Re<br>Z=%{z:.2f} Re<extra></extra>'
        ))

        # ===== ADD EARTH =====
        u = np.linspace(0, 2 * np.pi, 50)
        v = np.linspace(0, np.pi, 30)
        x_earth = np.outer(np.cos(u), np.sin(v))
        y_earth = np.outer(np.sin(u), np.sin(v))
        z_earth = np.outer(np.ones(np.size(u)), np.cos(v))

        fig.add_trace(go.Surface(
            x=x_earth, y=y_earth, z=z_earth,
            colorscale=[[0, '#1a4d7a'], [0.5, '#2e7cb8'], [1, '#2e7cb8']],
            showscale=False, name='Earth', hoverinfo='skip',
            lighting=dict(ambient=0.6, diffuse=0.8, specular=0.3)
        ))

        # ===== ADD GRIDLINES =====
        for lat in np.linspace(-np.pi/2, np.pi/2, 7):
            lons = np.linspace(0, 2*np.pi, 100)
            x_lat = np.cos(lat) * np.cos(lons)
            y_lat = np.cos(lat) * np.sin(lons)
            z_lat = np.sin(lat) * np.ones_like(lons)
            fig.add_trace(go.Scatter3d(
                x=x_lat, y=y_lat, z=z_lat, mode='lines',
                line=dict(color='rgba(255,255,255,0.3)', width=1),
                showlegend=False, hoverinfo='skip'
            ))

        for lon in np.linspace(0, 2*np.pi, 12, endpoint=False):
            lats = np.linspace(-np.pi/2, np.pi/2, 100)
            x_lon = np.cos(lats) * np.cos(lon)
            y_lon = np.cos(lats) * np.sin(lon)
            z_lon = np.sin(lats)
            fig.add_trace(go.Scatter3d(
                x=x_lon, y=y_lon, z=z_lon, mode='lines',
                line=dict(color='rgba(255,255,255,0.3)', width=1),
                showlegend=False, hoverinfo='skip'
            ))

        # ===== UPDATE LAYOUT WITH CALCULATED RANGES =====
        fig.update_layout(
            title=dict(text=""),
            scene=dict(
                xaxis_title="X<sub>GSM</sub> [R<sub>⊕</sub>]",
                yaxis_title="Y<sub>GSM</sub> [R<sub>⊕</sub>]",
                zaxis_title="Z<sub>GSM</sub> [R<sub>⊕</sub>]",
                xaxis=dict(range=x_lim),
                yaxis=dict(range=y_lim),
                zaxis=dict(range=z_lim),
            ),
            width=1200, height=800, hovermode='closest'
        )

        print(f"✓")

        # Build crossing info for HTML
        crossing_info = f"📍 Pass {pass_id} - {direction_name} Crossings: {len(crossings)}\n"
        crossing_info += f"   Wind Parameters (from {date_used}): Pdyn={Pdyn:.2f} nPa, Bz={Bz:+.2f} nT, Ma={Ma:.1f}, Dst={Dst:.0f}nT\n"
        crossing_info += f"   Plot Range: X=[{x_lim[0]:.1f}, {x_lim[1]:.1f}], Y=[{y_lim[0]:.1f}, {y_lim[1]:.1f}], Z=[{z_lim[0]:.1f}, {z_lim[1]:.1f}]\n\n"

        for c in crossings:
            if c.has_broad_window():
                crossing_info += f"     Broad window:  {c.broad_start.strftime('%Y-%m-%d %H:%M:%S')} to {c.broad_end.strftime('%Y-%m-%d %H:%M:%S')}\n"
                break

        for c in crossings:
            if c.has_zoom_window():
                crossing_info += f"     Zoom window:   {c.zoom_start.strftime('%Y-%m-%d %H:%M:%S')} to {c.zoom_end.strftime('%Y-%m-%d %H:%M:%S')}\n"
                break

        if crossings:
            crossing_info += f"     Time range: {min(c.time for c in crossings).strftime('%Y-%m-%d %H:%M:%S')} to {max(c.time for c in crossings).strftime('%Y-%m-%d %H:%M:%S')}\n"

        crossing_info += "-" * 80 + "\n"
        for i, c in enumerate(crossings, 1):
            crossing_info += f"  {i}. Crossing: {c.time.strftime('%Y-%m-%d %H:%M:%S')}\n"

        # Save HTML
        figs_dir = Path("figs_wind")
        figs_dir.mkdir(exist_ok=True)

        html_file = figs_dir / f"Magnetosphere_Pass_{pass_id:02d}_{direction_name}_{year:04d}-{month:02d}-{day:02d}T{hour:02d}{minute:02d}_WindData.html"

        plot_html = fig.to_html(include_plotlyjs='cdn')
        full_html = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Magnetosphere Pass {pass_id} - {direction_name} Crossings (Wind Data)</title>
    <style>
        body {{font-family: monospace; margin: 20px; background-color: #f5f5f5;}}
        .header {{background-color: white; padding: 20px; border-radius: 5px; margin-bottom: 20px; white-space: pre; line-height: 1.6;}}
        .plot-container {{background-color: white; padding: 20px; border-radius: 5px;}}
    </style>
</head>
<body>
    <div class="header">{crossing_info}</div>
    <div class="plot-container">{plot_html}</div>
</body>
</html>"""

        with open(str(html_file), 'w', encoding='utf-8') as f:
            f.write(full_html)

        print(f"     ✓ Saved: {html_file.name}")
        results['success'].append(f"Pass {pass_id} ({direction_name})")

    except Exception as e:
        print(f"     ✗ ERROR: {type(e).__name__}: {str(e)[:70]}")
        results['failed'].append({
            'pass': pass_id,
            'direction': direction_name,
            'error': f"{type(e).__name__}: {str(e)[:100]}"
        })

    print()

# Summary
end_time = datetime.now()
elapsed = end_time - start_time
print("=" * 80)
print("BATCH PROCESSING COMPLETE")
print("=" * 80)
print(f"\n⏱️  Time elapsed: {elapsed}")
print(f"\n✅ Successful: {len(results['success'])}/{len(failed_passes)}")
for item in results['success']:
    print(f"   • {item}")

if results['failed']:
    print(f"\n❌ Failed: {len(results['failed'])}")
    for item in results['failed']:
        print(f"   • Pass {item['pass']} ({item['direction']})")
        print(f"     → {item['error']}")
else:
    print(f"\n🎉 No failures!")

print("\n" + "=" * 80)
print(f"All HTML files saved to: figs_wind/")
print("=" * 80)

## Summary & Outputs

✅ **Generated visualizations are saved to `figs/` folder**

### Quick Reference

- **Equatorial Plane (XY):** Shows magnetopause nose distance, bow shock standoff, and Wind orbit in the plane perpendicular to Earth's dipole
- **Meridian Plane (XZ):** Shows field lines traced via T96+IGRF model, magnetosphere extent, and Wind's radial position

### Model Details

| Model | Description |
|-------|-------------|
| **Shue 1998** | Empirical magnetopause model; scales with solar wind dynamic pressure (Pdyn) and IMF Bz |
| **Conic Bow Shock** | Bow shock shape from Alfvén Mach number; distance depends on magnetopause nose and Ma |
| **T96** | Empirical magnetospheric field model; used for field line tracing |
| **IGRF** | International Geomagnetic Reference Field; Earth's intrinsic magnetic field |
| **Wind Ephemeris** | NASA WIND spacecraft position via SSCWeb service |
| **OMNI2** | Space weather data (Pdyn, Bz, By, Dst, Ma) |

---

**Questions?** Refer to the README.md and scripts/ folder for more details.

## Wind data and updated range

In [ ]:
# Physical constants
m_p = 1.6726e-27  # proton mass (kg)
k_b = 1.3807e-23  # Boltzmann constant (J/K)
e_c = 1.6022e-19  # elementary charge (C)
mu_0 = 1.2566e-6  # permeability of free space

DATA_DIR = r"D:\Data\Wind\bow_shock\data_used"

def get_mag_data_wind(filename_mag):
    """Read magnetic field data from Wind MFI CDF file."""
    time_shift = 6.21671442e13 + 1250. * 60. * 1e3

    cdf_file = cdflib.CDF(filename_mag)
    epoch_mag = cdf_file.varget("Epoch")
    unix_ms = epoch_mag - time_shift
    B_GSM = cdf_file.varget("BGSM")

    time_stamps_mag = pd.to_datetime(unix_ms, unit='ms')
    data_mag = np.concatenate((np.array([epoch_mag]).T, B_GSM), axis=1)
    df_mag = pd.DataFrame(data_mag, columns=['epoch', 'B_X [nT]', 'B_Y [nT]', 'B_Z [nT]'],
                          index=time_stamps_mag)
    df_mag['B [nT]'] = (df_mag['B_X [nT]']**2 + df_mag['B_Y [nT]']**2 + df_mag['B_Z [nT]']**2)**0.5

    df_mag.dropna(inplace=True)
    # Downsample to reduce processing time
    df_mag = df_mag.resample('5min').mean()

    return df_mag

def get_fc_data(filename_fc):
    """Read solar wind (Faraday Cup) data from Wind SWE CDF file."""
    cdf_file = cdflib.CDF(filename_fc)
    info = cdf_file.cdf_info()
    
    # Extract year from filename - handles both formats:
    # Old: swe_YYYY_YYYYMMDD_v??.cdf
    # New: wi_h1_swe_YYYYMMDD_v??.cdf or wi_h2_swe_YYYYMMDD_v??.cdf
    basename = os.path.basename(filename_fc)
    parts = basename.split('_')
    
    # Find the date part (YYYYMMDD format, 8 digits)
    year = None
    for part in parts:
        if part.isdigit() and len(part) == 8:
            year = int(part[:4])  # First 4 digits are year
            break
    
    if year is None:
        raise ValueError(f"Could not extract year from filename: {basename}")
    
    doy = cdf_file.varget('doy')
    df = pd.DataFrame(data=cdf_file.varget('Epoch'), columns=['epoch FC'])
    # Convert day-of-year to datetime
    df.index = pd.to_datetime(doy - 1, unit='D', origin=pd.Timestamp(year=year, month=1, day=1))
    for variable in info.zVariables:
        if variable in ['fit_flag', 'Proton_V_nonlin', 'Proton_W_nonlin', 'Proton_Np_nonlin', 'Alpha_Na_nonlin']:
            df[variable] = cdf_file.varget(variable)
    df.columns = ['epoch FC', 'FC QF', 'v_sw [m/s]', 'v_tp [m/s]', 'n_p [cm^-3]', 'n_a [cm^-3]']
    # Filter and validate data
    df.loc[df['v_sw [m/s]'] > 10000, 'v_sw [m/s]'] = np.nan
    df.loc[df['v_sw [m/s]'] < 0, 'v_sw [m/s]'] = np.nan
    df.loc[df['v_tp [m/s]'] > 1000, 'v_tp [m/s]'] = np.nan
    df.loc[df['v_tp [m/s]'] < 0, 'v_tp [m/s]'] = np.nan
    df.loc[df['n_p [cm^-3]'] > 1000, 'n_p [cm^-3]'] = np.nan
    df.loc[df['n_p [cm^-3]'] < 0, 'n_p [cm^-3]'] = np.nan
    # Convert velocities to m/s
    df['v_sw [m/s]'] = df['v_sw [m/s]'] * 1e3
    df['v_tp [m/s]'] = df['v_tp [m/s]'] * 1e3
    # Quality flag filter
    df = df[df['FC QF'] > 0]
    df.loc[df['v_tp [m/s]'] > 5e5, 'v_tp [m/s]'] = np.nan
    df['v_tp [m/s]'] = df['v_tp [m/s]'].interpolate()
    df.loc[df['n_a [cm^-3]'] > 100, 'n_a [cm^-3]'] = np.nan
    df['n_a [cm^-3]'] = df['n_a [cm^-3]'].interpolate()
    df['n_e [cm^-3]'] = df['n_p [cm^-3]'] + 2. * df['n_a [cm^-3]']
    return df

def calculate_temperature(v_tp_ms):
    """Calculate proton temperature from thermal speed (m/s) in eV."""
    return (m_p * v_tp_ms**2) / (2 * 1.602e-19)

def calculate_pdyn(n_p, v_sw_ms):
    """Calculate dynamic pressure in nPa."""
    v_sw_kms = v_sw_ms / 1000.0
    return 1.6726e-6 * n_p * v_sw_kms**2

def calculate_mach_number(v_sw_ms, bz_nt, by_nt, bx_nt, T_p_ev, n_p_cm3):
    """Calculate magnetosonic Mach number."""
    v_sw_kms = v_sw_ms / 1000.0
    B_nt = np.sqrt(bx_nt**2 + by_nt**2 + bz_nt**2)

    gamma = 5.0 / 3.0
    c_s_ms = np.sqrt(gamma * T_p_ev * e_c / m_p)
    c_s_kms = c_s_ms / 1000.0

    rho_kg_m3 = n_p_cm3 * 1e6 * m_p
    B_t = B_nt * 1e-9

    if rho_kg_m3 > 0:
        v_a_ms = B_t / np.sqrt(mu_0 * rho_kg_m3)
        v_a_kms = v_a_ms / 1000.0
    else:
        v_a_kms = 0

    c_ms_kms = np.sqrt(c_s_kms**2 + v_a_kms**2)

    if c_ms_kms > 0:
        ma = v_sw_kms / c_ms_kms
    else:
        ma = np.nan

    return ma

def parse_dst_file(filepath):
    """Parse Dst file in compact format."""
    dst_dict = {}

    try:
        with open(filepath, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or len(line) < 12:
                    continue

                try:
                    yy = int(line[3:5])
                    mm = int(line[5:7])
                    dd = int(line[8:10])

                    if yy < 50:
                        year = 2000 + yy
                    else:
                        year = 1900 + yy

                    values_str = line[10:].strip()
                    fields = values_str.split()

                    hourly_vals = []
                    for field in fields:
                        if field.startswith('-'):
                            hourly_vals.append(float(field))
                        else:
                            parts = field.split('-')
                            for i, part in enumerate(parts):
                                if part:
                                    if i > 0:
                                        hourly_vals.append(-float(part))
                                    else:
                                        hourly_vals.append(float(part))

                    dst_dict[(year, mm, dd)] = hourly_vals[:24]

                except (ValueError, IndexError):
                    continue

    except Exception as e:
        print(f"✗ Error parsing Dst file: {e}")

    return dst_dict

def get_dst(year, month, day, hour=0):
    """Get Dst value for a specific date/time."""
    dst_file = os.path.join(DATA_DIR, f"dst{year:04d}.txt")

    if not os.path.exists(dst_file):
        return np.nan

    dst_dict = parse_dst_file(dst_file)
    key = (year, month, day)

    if key not in dst_dict:
        return np.nan

    hourly_vals = dst_dict[key]
    if 0 <= hour < 24:
        return float(hourly_vals[hour])

    return np.nan

def get_wind_params(year, month, day, hour, minute, mfi_filepath=None, swe_filepath=None, 
                   mfi_tolerance=300, swe_tolerance=1800,
                   use_typical_values_mag=False, use_typical_values_swe=False,
                   mag_resample_minutes=1):
    """
    Get all Wind parameters at a specific date/time.
    Args:
        year, month, day, hour, minute: Date/time to extract
        mfi_filepath: Optional path to MFI CDF file (auto-finds if not provided)
        swe_filepath: Optional path to SWE CDF file (auto-finds if not provided)
        mfi_tolerance: Max seconds from target time for MFI data (default 300 = 5 min)
        swe_tolerance: Max seconds from target time for SWE data (default 1800 = 30 min)
        use_typical_values_mag: If True, use typical values if MFI data unavailable (default False)
        use_typical_values_swe: If True, use typical values if SWE data unavailable (default False)
        mag_resample_minutes: Downsample MFI data to this interval in minutes (default 1 min)
                             Set to 0 to disable downsampling
    Returns dict with: Pdyn, Dst, By, Bz, Bx, Ma, T_p, v_sw, n_p
    Or None if data not available
    """
    # Typical/default values for when data is missing
    TYPICAL_BX = -1.5  # nT
    TYPICAL_BY = 2.0   # nT
    TYPICAL_BZ = -2.0  # nT
    TYPICAL_V_SW = 450.0  # km/s
    TYPICAL_N_P = 7.0    # cm^-3
    TYPICAL_V_TP = 40000.0  # m/s (thermal speed)
    
    # Find or use provided MFI and SWE files
    if mfi_filepath is None or swe_filepath is None:
        mfi_files = [f for f in os.listdir(DATA_DIR) if f.startswith(f"mfi_{year:04d}")]
        swe_files = [f for f in os.listdir(DATA_DIR) if f.startswith(f"swe_{year:04d}")]
        if not mfi_files or not swe_files:
            if not use_typical_values_mag or not use_typical_values_swe:
                print(f"✗ CDF files not found for {year}")
                return None
        if mfi_filepath is None and mfi_files:
            mfi_filepath = os.path.join(DATA_DIR, mfi_files[0])
        if swe_filepath is None and swe_files:
            swe_filepath = os.path.join(DATA_DIR, swe_files[0])
    
    try:
        target_time = pd.Timestamp(year=year, month=month, day=day, hour=hour, minute=minute)
        
        # Extract magnetic field data
        if mfi_filepath and os.path.exists(mfi_filepath):
            df_mag = get_mag_data_wind(mfi_filepath)
            
            # Downsample if requested (reduces processing time for large files)
            if mag_resample_minutes > 0:
                print(f"  Downsampling MFI: {len(df_mag)} → ", end="")
                df_mag = df_mag.resample(f'{mag_resample_minutes}min').mean()
                df_mag = df_mag.dropna()  # Remove NaN values from resampling
                print(f"{len(df_mag)} records")
            
            time_diffs_mag = np.abs((df_mag.index - target_time).total_seconds())
            idx_mag = np.argmin(time_diffs_mag)
            if time_diffs_mag[idx_mag] <= mfi_tolerance:
                bx = df_mag['B_X [nT]'].iloc[idx_mag]
                by = df_mag['B_Y [nT]'].iloc[idx_mag]
                bz = df_mag['B_Z [nT]'].iloc[idx_mag]
            elif use_typical_values_mag:
                print(f"⚠️  MFI data too far ({time_diffs_mag[idx_mag]:.0f}s > {mfi_tolerance}s), using typical values")
                bx, by, bz = TYPICAL_BX, TYPICAL_BY, TYPICAL_BZ
            else:
                return None
        elif use_typical_values_mag:
            print(f"⚠️  MFI file not found, using typical values")
            bx, by, bz = TYPICAL_BX, TYPICAL_BY, TYPICAL_BZ
        else:
            return None
        
        # Extract solar wind data
        if swe_filepath and os.path.exists(swe_filepath):
            df_sw = get_fc_data(swe_filepath)
            time_diffs_sw = np.abs((df_sw.index - target_time).total_seconds())
            idx_sw = np.argmin(time_diffs_sw)
            if time_diffs_sw[idx_sw] <= swe_tolerance:
                v_sw = df_sw['v_sw [m/s]'].iloc[idx_sw]
                n_p = df_sw['n_p [cm^-3]'].iloc[idx_sw]
                v_tp = df_sw['v_tp [m/s]'].iloc[idx_sw]
            elif use_typical_values_swe:
                print(f"⚠️  SWE data too far ({time_diffs_sw[idx_sw]:.0f}s > {swe_tolerance}s), using typical values")
                v_sw = TYPICAL_V_SW
                n_p = TYPICAL_N_P
                v_tp = TYPICAL_V_TP
            else:
                return None
        elif use_typical_values_swe:
            print(f"⚠️  SWE file not found, using typical values")
            v_sw = TYPICAL_V_SW
            n_p = TYPICAL_N_P
            v_tp = TYPICAL_V_TP
        else:
            return None
        
        # Calculate derived parameters
        pdyn = calculate_pdyn(n_p, v_sw)
        t_p = calculate_temperature(v_tp)
        ma = calculate_mach_number(v_sw, bz, by, bx, t_p, n_p)
        dst = get_dst(year, month, day, hour)
        
        return {
            'Pdyn': pdyn,
            'Dst': dst,
            'By': by,
            'Bz': bz,
            'Bx': bx,
            'Ma': ma,
            'T_p': t_p,
            'v_sw': v_sw,
            'n_p': n_p,
        }
    except Exception as e:
        print(f"✗ Error extracting parameters: {e}")
        return None

In [ ]:
"""
BATCH PROCESSING: 18 FAILED PASSES WITH WIND DATA
Uses explicit file paths for passes with known file locations
Generates 3D visualizations with dynamic range adjustment based on Wind trajectory
"""
print("=" * 80)
print("BATCH PROCESSING: 18 FAILED PASSES WITH WIND DATA")
print("=" * 80)
print()
from datetime import datetime, timedelta
from pathlib import Path
import os
import pandas as pd
import numpy as np
import datetime as dt
import plotly.graph_objects as go

# List of 18 failed passes to process
failed_passes = [
    (3, 'out', 1995, 9, 16, 10, 17),
    (4, 'in', 1995, 11, 28, 8, 40),
    (5, 'in', 1995, 12, 20, 21, 40),
    (5, 'out', 1995, 12, 22, 13, 46),
    (6, 'out', 1996, 1, 14, 7, 10),
    (8, 'in', 1996, 4, 17, 14, 15),
    (9, 'in', 1996, 5, 9, 14, 0),
    (13, 'out', 1996, 11, 16, 19, 0),
    (15, 'out', 1997, 1, 1, 16, 30),
    (16, 'out', 1997, 6, 12, 13, 55),
    (19, 'out', 1997, 9, 9, 19, 21),
    (20, 'out', 1997, 10, 1, 12, 15),
    (22, 'in', 1998, 7, 1, 18, 18),
    (24, 'out', 1998, 8, 16, 22, 37),
    (31, 'in', 1999, 2, 4, 11, 27),
    (42, 'out', 1999, 11, 18, 6, 45),
    (55, 'in', 2001, 2, 20, 2, 45),
    (56, 'in', 2001, 9, 8, 12, 5),
]

# Explicit file paths for passes where we have confirmed files
# Format: (pass_id, direction): (mfi_filepath, swe_filepath)
explicit_files = {
    #(42, 'out'): (
    #    r"D:\Data\Wind\bow_shock\data_used\mfi_1999_19991118_v05.cdf",
    #    r"D:\Data\Wind\bow_shock\data_used\wi_h1_swe_19991118_v01.cdf"
    #),
    # Add Pass 56 files here when you have them:
    (56, 'in'): (
        r"D:\Data\Wind\bow_shock\data_used\wi_h2_mfi_20010908_v05.cdf",
        r"D:\Data\Wind\bow_shock\data_used\wi_h1_swe_20010908_v01.cdf"
    ),
}

# TEMPORARY: Only process passes with explicit files for now
failed_passes = [p for p in failed_passes if (p[0], p[1]) in explicit_files]

DATA_DIR = r"D:\Data\Wind\bow_shock\data_used"
print(f"Found {len(failed_passes)} failed passes to process\n")
results = {'success': [], 'failed': []}
start_time = datetime.now()

# Process each failed pass
for pass_idx, (pass_id, direction, year, month, day, hour, minute) in enumerate(failed_passes, 1):
    direction_name = "Inbound" if direction == 'in' else "Outbound"
    print(f"[{pass_idx:2d}/{len(failed_passes)}] Pass {pass_id:2d} ({direction_name:7s}) {year}-{month:02d}-{day:02d}T{hour:02d}:{minute:02d}...")
    
    # Get crossings for this pass from database
    all_crossings = db.get_crossings_for_pass(pass_id)
    crossings = [c for c in all_crossings if c.crossing_direction == direction]
    
    if not crossings:
        print(f"     ⊘ No {direction_name.lower()} crossings - skipping\n")
        continue
    
    try:
        # Find reference time for visualization
        t0 = None
        for c in crossings:
            if c.has_zoom_window():
                t0 = c.zoom_start + (c.zoom_end - c.zoom_start) / 2
                break
        if t0 is None:
            t0 = crossings[0].time
        
        print(f"     Extracting Wind parameters...", end=" ")
        
        # Get explicit file paths if available, otherwise auto-find
        if (pass_id, direction) in explicit_files:
            mfi_file, swe_file = explicit_files[(pass_id, direction)]
            params = get_wind_params(year, month, day, hour, minute,
                                    mfi_filepath=mfi_file,
                                    swe_filepath=swe_file,
                                    mfi_tolerance=21600,  # ±6 hours
                                    swe_tolerance=21600)  # ±6 hours
        else:
            params = get_wind_params(year, month, day, hour, minute,
                                    mfi_tolerance=21600,  # ±6 hours
                                    swe_tolerance=21600)  # ±6 hours
        
        date_used = f"{year}-{month:02d}-{day:02d}"
        if params is None:
            raise Exception("Could not extract Wind parameters (even with ±6h tolerance)")
        
        Pdyn = params['Pdyn']
        Dst = params['Dst']
        By = params['By']
        Bz = params['Bz']
        Ma = params['Ma']
        
        if not (np.isfinite(Pdyn) and np.isfinite(Bz)):
            raise Exception(f"Invalid params: Pdyn={Pdyn}, Bz={Bz}")
        
        print(f"✓")
        print(f"     Parameters: Pdyn={Pdyn:.2f}nPa, Bz={Bz:.2f}nT, Ma={Ma:.1f}, Dst={Dst:.0f}nT")
        
        # Compute magnetospheric boundaries
        print(f"     Computing boundaries...", end=" ")
        parmod = t96_parmod(Pdyn, Dst, By, Bz)
        xmp, ymp, r0mp, alphamp = shue98_magnetopause_rtheta(Pdyn, Bz, ntheta=361)
        xbs, ybs, rbs0, delta_bs = bowshock_rtheta_from_Ma(r0mp, Ma, ntheta=361)
        print(f"✓")
        
        # Verify boundaries are valid
        if not (np.isfinite(xmp).all() and np.isfinite(ymp).all()):
            raise Exception("Magnetopause calculation failed")
        if not (np.isfinite(xbs).all() and np.isfinite(ybs).all()):
            raise Exception("Bow shock calculation failed")
        
        # Trace field lines
        print(f"     Tracing field lines...", end=" ")
        seeds = []
        r_seed = 2.5
        for lat in np.linspace(-np.pi/2, np.pi/2, 6):
            for lon in np.linspace(0, 2*np.pi, 12, endpoint=False):
                x = r_seed * np.cos(lat) * np.cos(lon)
                y = r_seed * np.cos(lat) * np.sin(lon)
                z = r_seed * np.sin(lat)
                seeds.append((x, y, z))
        
        for r in [3.5, 5.0]:
            for angle in np.linspace(0, np.pi/2, 6):
                for sign in [1, -1]:
                    seeds.append((-r * np.cos(angle), 0, sign * r * np.sin(angle)))
        
        field_lines = []
        ut = unix_seconds(t0)
        for s in seeds:
            try:
                result = trace_field_line(s, parmod, ut, rlim=60.0, r0=1.0)
                if result is not None:
                    X, Y, Z, ps = result
                    field_lines.append((X, Y, Z))
            except:
                pass  # Skip seeds that fail to trace
        print(f"✓ ({len(field_lines)} lines)")
        
        # Fetch Wind trajectory
        print(f"     Fetching Wind trajectory...", end=" ")
        start = (t0 - dt.timedelta(hours=12)).isoformat()
        stop = (t0 + dt.timedelta(hours=12)).isoformat()
        times_w, xyz_w = get_wind_gsm_from_ssc(start, stop, observatory_id="wind")
        print(f"✓ ({len(times_w)} points)")
        
        # Calculate plot range based on Wind trajectory
        print(f"     Calculating plot range...", end=" ")
        if len(xyz_w) > 0:
            x_max_dist = max(abs(xyz_w[:, 0].min()), abs(xyz_w[:, 0].max()))
            y_max_dist = max(abs(xyz_w[:, 1].min()), abs(xyz_w[:, 1].max()))
            z_max_dist = max(abs(xyz_w[:, 2].min()), abs(xyz_w[:, 2].max()))
            
            # Ensure minimum range and add 20% padding
            x_max_dist = max(x_max_dist, 5) * 1.2
            y_max_dist = max(y_max_dist, 5) * 1.2
            z_max_dist = max(z_max_dist, 5) * 1.2
            
            x_lim = [-x_max_dist, x_max_dist]
            y_lim = [-y_max_dist, y_max_dist]
            z_lim = [-z_max_dist, z_max_dist]
        else:
            x_lim = y_lim = z_lim = [-40, 40]
        print(f"✓ X=[{x_lim[0]:.1f}, {x_lim[1]:.1f}]")
        
        # Create Plotly figure
        print(f"     Building visualization...", end=" ")
        fig = go.Figure()
        
        # ===== ADD MAGNETOPAUSE =====
        angles_mp = np.linspace(0, 2*np.pi, 100)
        xmp_flat, ymp_flat, zmp_flat = [], [], []
        for angle in angles_mp:
            xmp_flat.extend(xmp)
            ymp_flat.extend(ymp * np.cos(angle))
            zmp_flat.extend(ymp * np.sin(angle))
        
        faces_i, faces_j, faces_k = [], [], []
        n_angles, n_radii = len(angles_mp), len(xmp)
        for i in range(n_angles - 1):
            for j in range(n_radii - 1):
                v0 = i * n_radii + j
                v1 = i * n_radii + (j + 1)
                v2 = ((i + 1) % n_angles) * n_radii + j
                v3 = ((i + 1) % n_angles) * n_radii + (j + 1)
                faces_i.extend([v0, v0, v2])
                faces_j.extend([v1, v2, v3])
                faces_k.extend([v2, v3, v1])
        
        fig.add_trace(go.Mesh3d(
            x=xmp_flat, y=ymp_flat, z=zmp_flat,
            i=faces_i, j=faces_j, k=faces_k,
            color='#326ba8', name='Magnetopause', opacity=0.35, showlegend=True, hoverinfo='skip'
        ))
        
        # ===== ADD BOW SHOCK =====
        angles_bs = np.linspace(0, 2*np.pi, 80)
        xbs_surf = np.tile(xbs, (len(angles_bs), 1))
        ybs_surf = np.zeros((len(angles_bs), len(xbs)))
        zbs_surf = np.zeros((len(angles_bs), len(xbs)))
        for i, angle in enumerate(angles_bs):
            ybs_surf[i, :] = ybs * np.cos(angle)
            zbs_surf[i, :] = ybs * np.sin(angle)
        
        fig.add_trace(go.Surface(
            x=xbs_surf, y=ybs_surf, z=zbs_surf,
            colorscale=[[0, '#32a852'], [1, '#32a852']], showscale=False,
            name='Bow shock', opacity=0.2, hoverinfo='skip'
        ))
        
        # ===== ADD FIELD LINES =====
        for i, (X, Y, Z) in enumerate(field_lines):
            fig.add_trace(go.Scatter3d(
                x=X, y=Y, z=Z, mode='lines', name='Field lines' if i == 0 else '',
                line=dict(color='#1f77b4', width=3), opacity=0.8,
                showlegend=(i == 0), hoverinfo='skip'
            ))
        
        # ===== ADD WIND TRAJECTORY =====
        fig.add_trace(go.Scatter3d(
            x=xyz_w[:, 0], y=xyz_w[:, 1], z=xyz_w[:, 2], mode='lines',
            name='Wind trajectory', line=dict(color='black', width=4), opacity=0.8, hoverinfo='skip'
        ))
        
        # ===== ADD TIME WINDOWS =====
        has_broad = False
        for c in crossings:
            if c.has_broad_window():
                broad_indices = [i for i, t in enumerate(times_w) if c.broad_start <= t <= c.broad_end]
                if broad_indices:
                    idx_start, idx_end = broad_indices[0], broad_indices[-1]
                    fig.add_trace(go.Scatter3d(
                        x=xyz_w[idx_start:idx_end+1, 0], y=xyz_w[idx_start:idx_end+1, 1], z=xyz_w[idx_start:idx_end+1, 2],
                        mode='lines', name='Broad window' if not has_broad else '',
                        line=dict(color='orange', width=5), opacity=0.6,
                        showlegend=not has_broad, hoverinfo='skip'
                    ))
                    has_broad = True
        
        has_zoom = False
        for c in crossings:
            if c.has_zoom_window():
                zoom_indices = [i for i, t in enumerate(times_w) if c.zoom_start <= t <= c.zoom_end]
                if zoom_indices:
                    idx_start, idx_end = zoom_indices[0], zoom_indices[-1]
                    fig.add_trace(go.Scatter3d(
                        x=xyz_w[idx_start:idx_end+1, 0], y=xyz_w[idx_start:idx_end+1, 1], z=xyz_w[idx_start:idx_end+1, 2],
                        mode='lines', name='Zoom window' if not has_zoom else '',
                        line=dict(color='red', width=5), opacity=0.8,
                        showlegend=not has_zoom, hoverinfo='skip'
                    ))
                    has_zoom = True
        
        # ===== ADD CROSSING MARKERS =====
        crossing_color = 'red' if direction == 'in' else 'darkblue'
        crossing_symbol = 'diamond' if direction == 'in' else 'diamond-open'
        crossing_x, crossing_y, crossing_z, crossing_times = [], [], [], []
        
        for c in crossings:
            time_diffs = [abs((t - c.time).total_seconds()) for t in times_w]
            if time_diffs:
                idx = np.argmin(time_diffs)
                crossing_x.append(xyz_w[idx, 0])
                crossing_y.append(xyz_w[idx, 1])
                crossing_z.append(xyz_w[idx, 2])
                crossing_times.append(c.time.strftime('%Y-%m-%d %H:%M:%S'))
        
        fig.add_trace(go.Scatter3d(
            x=crossing_x, y=crossing_y, z=crossing_z, mode='markers',
            name=f'{direction_name} crossings',
            marker=dict(size=8, color=crossing_color, symbol=crossing_symbol,
                       line=dict(color='black', width=2), opacity=0.9),
            text=crossing_times,
            hovertemplate='<b>%{text}</b><br>X=%{x:.2f} Re<br>Y=%{y:.2f} Re<br>Z=%{z:.2f} Re<extra></extra>'
        ))
        
        # ===== ADD EARTH =====
        u = np.linspace(0, 2 * np.pi, 50)
        v = np.linspace(0, np.pi, 30)
        x_earth = np.outer(np.cos(u), np.sin(v))
        y_earth = np.outer(np.sin(u), np.sin(v))
        z_earth = np.outer(np.ones(np.size(u)), np.cos(v))
        
        fig.add_trace(go.Surface(
            x=x_earth, y=y_earth, z=z_earth,
            colorscale=[[0, '#1a4d7a'], [0.5, '#2e7cb8'], [1, '#2e7cb8']],
            showscale=False, name='Earth', hoverinfo='skip',
            lighting=dict(ambient=0.6, diffuse=0.8, specular=0.3)
        ))
        
        # ===== ADD GRIDLINES =====
        for lat in np.linspace(-np.pi/2, np.pi/2, 7):
            lons = np.linspace(0, 2*np.pi, 100)
            x_lat = np.cos(lat) * np.cos(lons)
            y_lat = np.cos(lat) * np.sin(lons)
            z_lat = np.sin(lat) * np.ones_like(lons)
            fig.add_trace(go.Scatter3d(
                x=x_lat, y=y_lat, z=z_lat, mode='lines',
                line=dict(color='rgba(255,255,255,0.3)', width=1),
                showlegend=False, hoverinfo='skip'
            ))
        
        for lon in np.linspace(0, 2*np.pi, 12, endpoint=False):
            lats = np.linspace(-np.pi/2, np.pi/2, 100)
            x_lon = np.cos(lats) * np.cos(lon)
            y_lon = np.cos(lats) * np.sin(lon)
            z_lon = np.sin(lats)
            fig.add_trace(go.Scatter3d(
                x=x_lon, y=y_lon, z=z_lon, mode='lines',
                line=dict(color='rgba(255,255,255,0.3)', width=1),
                showlegend=False, hoverinfo='skip'
            ))
        
        # ===== UPDATE LAYOUT WITH CALCULATED RANGES =====
        fig.update_layout(
            title=dict(text=""),
            scene=dict(
                xaxis_title="X<sub>GSM</sub> [R<sub>⊕</sub>]",
                yaxis_title="Y<sub>GSM</sub> [R<sub>⊕</sub>]",
                zaxis_title="Z<sub>GSM</sub> [R<sub>⊕</sub>]",
                xaxis=dict(range=x_lim),
                yaxis=dict(range=y_lim),
                zaxis=dict(range=z_lim),
            ),
            width=1200, height=800, hovermode='closest'
        )
        
        print(f"✓")
        
        # Build crossing info for HTML
        crossing_info = f"📍 Pass {pass_id} - {direction_name} Crossings: {len(crossings)}\n"
        crossing_info += f"   Wind Parameters (from {date_used}): Pdyn={Pdyn:.2f} nPa, Bz={Bz:+.2f} nT, Ma={Ma:.1f}, Dst={Dst:.0f}nT\n"
        crossing_info += f"   Plot Range: X=[{x_lim[0]:.1f}, {x_lim[1]:.1f}], Y=[{y_lim[0]:.1f}, {y_lim[1]:.1f}], Z=[{z_lim[0]:.1f}, {z_lim[1]:.1f}]\n\n"
        
        for c in crossings:
            if c.has_broad_window():
                crossing_info += f"     Broad window:  {c.broad_start.strftime('%Y-%m-%d %H:%M:%S')} to {c.broad_end.strftime('%Y-%m-%d %H:%M:%S')}\n"
                break
        
        for c in crossings:
            if c.has_zoom_window():
                crossing_info += f"     Zoom window:   {c.zoom_start.strftime('%Y-%m-%d %H:%M:%S')} to {c.zoom_end.strftime('%Y-%m-%d %H:%M:%S')}\n"
                break
        
        if crossings:
            crossing_info += f"     Time range: {min(c.time for c in crossings).strftime('%Y-%m-%d %H:%M:%S')} to {max(c.time for c in crossings).strftime('%Y-%m-%d %H:%M:%S')}\n"
        
        crossing_info += "-" * 80 + "\n"
        for i, c in enumerate(crossings, 1):
            crossing_info += f"  {i}. Crossing: {c.time.strftime('%Y-%m-%d %H:%M:%S')}\n"
        
        # Save HTML
        figs_dir = Path("figs_wind")
        figs_dir.mkdir(exist_ok=True)
        
        html_file = figs_dir / f"Magnetosphere_Pass_{pass_id:02d}_{direction_name}_{year:04d}-{month:02d}-{day:02d}T{hour:02d}{minute:02d}_WindData.html"
        
        plot_html = fig.to_html(include_plotlyjs='cdn')
        full_html = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Magnetosphere Pass {pass_id} - {direction_name} Crossings (Wind Data)</title>
    <style>
        body {{font-family: monospace; margin: 20px; background-color: #f5f5f5;}}
        .header {{background-color: white; padding: 20px; border-radius: 5px; margin-bottom: 20px; white-space: pre; line-height: 1.6;}}
        .plot-container {{background-color: white; padding: 20px; border-radius: 5px;}}
    </style>
</head>
<body>
    <div class="header">{crossing_info}</div>
    <div class="plot-container">{plot_html}</div>
</body>
</html>"""
        
        with open(str(html_file), 'w', encoding='utf-8') as f:
            f.write(full_html)
        
        print(f"     ✓ Saved: {html_file.name}")
        results['success'].append(f"Pass {pass_id} ({direction_name})")
        
    except Exception as e:
        print(f"     ✗ ERROR: {type(e).__name__}: {str(e)[:70]}")
        results['failed'].append({
            'pass': pass_id,
            'direction': direction_name,
            'error': f"{type(e).__name__}: {str(e)[:100]}"
        })
    
    print()

# Summary
end_time = datetime.now()
elapsed = end_time - start_time
print("=" * 80)
print("BATCH PROCESSING COMPLETE")
print("=" * 80)
print(f"\n⏱️  Time elapsed: {elapsed}")
print(f"\n✅ Successful: {len(results['success'])}/{len(failed_passes)}")
for item in results['success']:
    print(f"   • {item}")
if results['failed']:
    print(f"\n❌ Failed: {len(results['failed'])}")
    for item in results['failed']:
        print(f"   • Pass {item['pass']} ({item['direction']})")
        print(f"     → {item['error']}")
else:
    print(f"\n🎉 No failures!")
print("\n" + "=" * 80)
print(f"All HTML files saved to: figs_wind/")
print("=" * 80)